# Global Financial Inclusion Analysis
## *Predictive Modeling for Targeted Outreach Strategy*

---

<div align="center">

### **VALERIE JERONO**
#### Student ID: 222331
#### MSc Data Science and Analytics • August 2025 Cohort
#### Data Mining and Statistical Reporting (DMSR)

---

*Using the Global Findex Database (World Bank, 2024)*  
*Bridging the Gap: From 1.4 Billion Unbanked to Data-Driven Financial Inclusion*

</div>

---


## Introduction

Across the globe, the challenge of financial exclusion persists: **1.4 billion adults remain without a financial account**. The absence of this basic entry point to savings, payments, credit, and insurance does not simply reflect inequality — it reinforces it. For policymakers, development practitioners, and financial institutions, the question is no longer *whether* financial inclusion matters, but rather: *who* remains excluded, *where* they are, and *how* outreach can be designed to truly reach them.

---
---

This notebook takes that challenge head-on. Using the **Global Findex 2024 dataset**, we developed a **machine learning pipeline** to predict account ownership and uncover the drivers of exclusion. The aim was twofold: to build models that are **interpretable enough for policy use**, yet **powerful enough to capture the complexity** of financial behaviour across diverse populations.


Four models — Logistic Regression, Random Forest, Gradient Boosting, and SVM — were trained and evaluated. The standout performer was 🌲 **Random Forest**, combining strong predictive power with clear feature insights. Beyond the raw metrics, the analysis sheds light on the **systematic patterns of exclusion**: why some individuals remain unbanked and which factors — from business loans to digital payments — most influence inclusion.

--- 
---
To bridge research with practice, the findings have been translated into a **deployed web-based application**:
👉 **[finscopee.streamlit.app](https://finscopee.streamlit.app/)**

This tool allows policymakers and practitioners to move from static reports to **dynamic, evidence-based targeting**, helping ensure scarce resources reach the people who need them most.



In [ ]:

# Example usage:
"""
# Run individual steps
df_encoded, encoders, log1 = encode_categorical_variables(df)
df_digital, log2 = create_digital_engagement_score(df_encoded)
df_financial, log3 = create_financial_activity_score(df_digital)
df_govt, log4 = create_government_services_score(df_financial)
df_interaction, log5 = create_interaction_features(df_govt)
df_flags, log6 = create_strategic_binary_flags(df_interaction)
df_final, log7 = optimize_feature_count(df_flags, max_features=30)

# Or run complete pipeline
df_engineered, label_encoders, feature_logs = run_complete_feature_engineering(df, max_features=30)
"""


## **Abstract**


This study leverages the **Global Findex 2024 dataset** to address one of the world’s most pressing development challenges: financial exclusion. 
Using a dataset of **8,476 individuals** and 24 engineered features, we developed a **machine learning pipeline** to predict account ownership with high accuracy.

---

Among the models tested, the **Random Forest classifier emerged as the strongest performer**, achieving **89.6% accuracy**, **AUC-ROC of 0.9607**, and **AUC-PR of 0.9743**. Importantly, the model demonstrated excellent calibration, reaching **97% accuracy on medium-confidence predictions** and **98%+ accuracy on high-confidence predictions**, making it highly reliable for targeted outreach strategies.


Feature importance analysis revealed that **business loan access, emergency funds, and digital payment adoption** are the strongest predictors of account ownership. This finding underscores that **entrepreneurship, financial resilience, and digital infrastructure** are critical gateways to inclusion.

---

Beyond technical performance, the project translates predictive power into practice through a **deployed web-based tool** 👉 [**finscopee.streamlit.app**](https://finscopee.streamlit.app/). 

Policymakers and practitioners can now input profile data to receive:

* The **probability of financial exclusion**,
* A **ranked explanation of drivers**, and
* A framework for **tiered, data-driven interventions**.

By blending robust modelling with practical deployment, this work demonstrates how **data science can move financial inclusion from reactive policy to proactive strategy** — ensuring scarce resources reach the people who need them most.


In [3]:
# --- Core ---
import warnings        # Suppress/control warnings
import time            # Track execution time
import pickle          # Save/load models/data
import os

# --- Data ---
import pandas as pd    # Data manipulation
import numpy as np     # Numerical ops
import requests        # Fetch data (APIs/URLs)

# --- Visuals ---
import matplotlib.pyplot as plt   # Basic plots
import seaborn as sns             # Statistical plots
plt.style.use('default')
sns.set_palette("Set2")

# --- Preprocessing ---
from sklearn.impute import SimpleImputer, KNNImputer  # Fill missing values
from sklearn.preprocessing import LabelEncoder, StandardScaler 
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
# Encode categorical & scale features

# --- Models ---
from sklearn.linear_model import LogisticRegression  
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier  
from sklearn.svm import SVC

# --- Evaluation ---
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# --- Training/Validation ---
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV


### Set the environment.

* **Core:** warnings, time, pickle
* **Data:** pandas, numpy, requests
* **Visuals:** matplotlib, seaborn
* **Preprocessing:** imputers, encoders, scalers
* **Models:** logistic, random forest, boosting, SVM
* **Evaluation:** reports, confusion matrix, ROC
* **Validation:** train/test split, CV, grid search


In [4]:
import pandas as pd
import requests

# Fetch the data.
df = pd.read_csv("cleaned_findex_data.csv")

df.head()

,region,income_group,demo_group,demo_subgroup,has_account,borrowed_any,credit_card,biz_loan_source,biz_loan,loan_purpose_group,...,digital_pay,digital_pay_acc,govt_payment_recv,fin_resilience,emergency_funds,prefer_digital_acc,prefer_digital_fin,prefer_digital,saved_any,region_clean
0,South Asia (excluding high income),Low income,all,all,0.090050,0.430756,0.011517,0.028194,0.028194,0.039803,...,0.170049,0.039621,0.038429,0.037202,0.008257,0.036158,0.029721,0.097307,0.193199,South Asia
1,Europe & Central Asia (excluding high income),Upper middle income,all,all,0.282681,0.428768,0.084186,0.085616,0.085616,0.185552,...,0.275174,0.181174,0.132580,0.238534,0.105931,0.167175,0.126082,0.295811,0.303361,Europe & Central Asia
2,Middle East & North Africa (excluding high inc...,Lower middle income,all,all,0.332861,0.419846,0.028715,0.043257,0.043257,0.056418,...,0.199535,0.061703,0.057444,0.141612,0.011603,0.128058,0.089581,0.177632,0.232552,Middle East & North Africa
3,Sub-Saharan Africa (excluding high income),Lower middle income,all,all,0.392035,0.518253,0.119146,0.159156,0.159156,0.169686,...,0.278687,0.146427,0.115377,0.176155,0.154860,0.157954,0.107940,0.220488,0.420917,Sub-Saharan Africa
4,Latin America & Caribbean (excluding high income),Upper middle income,all,all,0.331302,0.391471,0.197019,0.038007,0.038007,0.225686,...,0.333847,0.184097,0.159569,0.205183,0.219409,0.190155,0.142711,0.264239,0.318876,Latin America & Caribbean


### 2. Dataset Overview

- Provides quick inspection: shape, columns, datatypes.

- Confirms dataset integrity and that preprocessing was successful.

In [5]:
print("Dataset Overview:")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(df.info())


Dataset Overview:
Shape: (8311, 30)
Columns: ['region', 'income_group', 'demo_group', 'demo_subgroup', 'has_account', 'borrowed_any', 'credit_card', 'biz_loan_source', 'biz_loan', 'loan_purpose_group', 'loan_purpose', 'saved_old_age', 'saved_for_purchase', 'saved_no_purpose', 'mobile_pay_s_r', 'digital_payment_other', 'mobile_payment', 'mobile_payment_bill', 'govt_digital_pay', 'govt_digital_pay_acc', 'digital_pay', 'digital_pay_acc', 'govt_payment_recv', 'fin_resilience', 'emergency_funds', 'prefer_digital_acc', 'prefer_digital_fin', 'prefer_digital', 'saved_any', 'region_clean']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8311 entries, 0 to 8310
Data columns (total 30 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   region                 8311 non-null   object 
 1   income_group           8311 non-null   object 
 2   demo_group             8311 non-null   object 
 3   demo_subgroup          8311 non-null   o

### 3. Target Variable Analysis (has_account)

- **has_account** = target variable (binary indicator of account ownership).

- Summarises distribution → average 61.1% ownership across the sample.

- Sets the baseline for classification task.

In [6]:
# Target variable analysis
print(f"\nTarget Variable (has_account) Distribution:")
print(df['has_account'].describe())
print(f"\nAccount ownership rate: {df['has_account'].mean():.3f}")

# Key statistics by region and income group
print("\nAccount Ownership by Region:")
region_stats = df.groupby('region')['has_account'].agg(['count', 'mean', 'std']).round(3)
print(region_stats)

print("\nAccount Ownership by Income Group:")
income_stats = df.groupby('income_group')['has_account'].agg(['count', 'mean', 'std']).round(3)
print(income_stats)



Target Variable (has_account) Distribution:
count    8311.000000
mean        0.611070
std         0.282490
min         0.004049
25%         0.374749
50%         0.624107
75%         0.880940
max         1.000000
Name: has_account, dtype: float64

Account ownership rate: 0.611

Account Ownership by Region:
                                                    count   mean    std
region                                                                 
East Asia & Pacific (excluding high income)           521  0.568  0.272
Europe & Central Asia (excluding high income)        1139  0.554  0.221
High income                                          2938  0.858  0.173
Latin America & Caribbean (excluding high income)     970  0.480  0.202
Middle East & North Africa (excluding high income)    558  0.382  0.230
South Asia (excluding high income)                    352  0.483  0.253
Sub-Saharan Africa (excluding high income)           1833  0.427  0.224

Account Ownership by Income Group:
        

### Descriptive Statistics by Groups

- By Region: highlights disparities (e.g., High income = 85.8%, Sub-Saharan Africa = 42.7%).

- By Income Group: strong gradient (High income = 87.0%, Low income = 37.4%).

- Provides contextual baselines before predictive modeling.

In [7]:
def analyze_target(df, target_col="has_account", threshold=0.5):
    """
    Quick check of target variable:
    - Shows value distribution
    - Detects imbalance
    """
    print(f"\n🎯 TARGET VARIABLE: {target_col}")
    print("=" * 35)

    # If continuous, binarize (0/1)
    if df[target_col].nunique() > 2:
        print("⚠️ Detected continuous target. Binarizing with threshold =", threshold)
        df[target_col] = (df[target_col] >= threshold).astype(int)

    # Distribution
    dist = df[target_col].value_counts()
    total = len(df)
    for val, count in dist.items():
        print(f"Class {val}: {count:,} ({count/total*100:.1f}%)")

    # Imbalance ratio
    imbalance_ratio = dist.max() / dist.min()
    print(f"\nImbalance ratio: {imbalance_ratio:.1f}:1")

    if imbalance_ratio > 5:
        print("⚠️ Dataset is imbalanced — consider resampling.")
    else:
        print("✅ Distribution is balanced enough.")
    
    return df

# Run
df_fixed = analyze_target(df, "has_account")



🎯 TARGET VARIABLE: has_account
⚠️ Detected continuous target. Binarizing with threshold = 0.5
Class 1: 5,141 (61.9%)
Class 0: 3,170 (38.1%)

Imbalance ratio: 1.6:1
✅ Distribution is balanced enough.


## Feature Engineering — Pipeline Explanation

This pipeline is designed to **reduce a dataset to 30 or fewer features** while keeping the most impactful information.
It balances **encoding, composite scores, and feature reduction**.

---

#### 1. **Encode Essential Categorical Variables**

* Columns: `region`, `income_group`, `demo_group`.
* Each is encoded into numerical values using **Label Encoding**.
* Original categorical columns are **dropped** to save space.
  Example: `region → region_encoded`.

---

#### 2. **Create Composite Scores (Feature Grouping)**

Instead of keeping many binary columns, we combine them into **scores**:

* **Digital Engagement Score**

  * From: `mobile_payment`, `mobile_payment_bill`, `digital_pay`, `digital_pay_acc`, `digital_payment_other`, `prefer_digital`.
  * Represents **overall digital adoption**.
  * Original columns dropped.

* **Financial Activity Score**

  * From: `borrowed_any`, `credit_card`, `saved_any`.
  * Shows **core financial behaviours**.
  * Originals kept (likely important for prediction).

* **Government Services Score**

  * From: `govt_digital_pay`, `govt_digital_pay_acc`, `govt_payment_recv`.
  * Reflects **usage of digital government services**.
  * Originals dropped.

---

#### 3. **Create Key Interaction Feature**

* Combines **income level** (`income_group_encoded`) with **digital adoption** (`digital_engagement_score`).
* Captures how **income affects digital engagement**.
   Feature: `income_digital_interaction`.

---

#### 4. **Add Strategic Binary Flags**

Binary flags are simple but powerful signals:

* **High Financial Activity**

  * `1` if `financial_activity_score` is above the 70th percentile.
  * Helps identify **financially active segments**.

* **Digital Native**

  * `1` if `digital_engagement_score` > 0.5.
  * Captures **digitally inclined users**.

---

#### 5. **Final Optimization (Feature Reduction)**

* If the number of features **exceeds 30**, the lowest-variance numeric features are dropped.
* Encoded columns and target variables are **protected** (never dropped).

---


In [8]:

def simplified_feature_engineering(df, max_features=30):
    """
    Simplified feature engineering targeting exactly 30 features or less
    Focus on the most impactful transformations
    """
    print(f"\n🔧 SIMPLIFIED FEATURE ENGINEERING")
    print("=" * 50)
    print(f"Starting with {df.shape[1]} columns, targeting max {max_features} columns")
    
    model_df = df.copy()
    feature_log = []
    
    # 1. ENCODE ONLY KEY CATEGORICAL VARIABLES
    print("\n1. Encoding essential categorical variables...")
    # Focus on the most important categorical variables
    key_categorical = ['region', 'income_group', 'demo_group']
    label_encoders = {}
    
    for col in key_categorical:
        if col in model_df.columns:
            le = LabelEncoder()
            model_df[col + '_encoded'] = le.fit_transform(model_df[col].astype(str))
            label_encoders[col] = le
            # Drop original categorical column to save space
            model_df = model_df.drop(col, axis=1)
            print(f"  ✅ {col} → {col}_encoded ({len(le.classes_)} categories)")
            feature_log.append(f"Encoded {col}")
    
    # 2. CREATE COMPOSITE SCORES (INSTEAD OF INDIVIDUAL FEATURES)
    print("\n2. Creating composite engagement scores...")
    
    # Digital Engagement Score - combine all digital payment features
    digital_cols = ['mobile_payment', 'mobile_payment_bill', 'digital_pay', 
                   'digital_pay_acc', 'digital_payment_other', 'prefer_digital']
    available_digital = [col for col in digital_cols if col in model_df.columns]
    
    if available_digital:
        model_df['digital_engagement_score'] = model_df[available_digital].mean(axis=1)
        # Drop individual digital columns to save space
        model_df = model_df.drop(available_digital, axis=1)
        print(f"  ✅ Digital engagement score from {len(available_digital)} columns")
        feature_log.append(f"Digital engagement score (combined {len(available_digital)} features)")
    
    # Financial Activity Score - combine core financial behaviors
    financial_cols = ['borrowed_any', 'credit_card', 'saved_any']
    available_financial = [col for col in financial_cols if col in model_df.columns]
    
    if available_financial:
        model_df['financial_activity_score'] = model_df[available_financial].mean(axis=1)
        # Keep the original columns as they're likely important for the target
        print(f"  ✅ Financial activity score from {len(available_financial)} columns")
        feature_log.append(f"Financial activity score (from {len(available_financial)} features)")
    
    # Government Services Score
    govt_cols = ['govt_digital_pay', 'govt_digital_pay_acc', 'govt_payment_recv']
    available_govt = [col for col in govt_cols if col in model_df.columns]
    
    if available_govt:
        model_df['govt_services_score'] = model_df[available_govt].mean(axis=1)
        # Drop individual govt columns
        model_df = model_df.drop(available_govt, axis=1)
        print(f"  ✅ Government services score from {len(available_govt)} columns")
        feature_log.append(f"Government services score (combined {len(available_govt)} features)")
    
    # 3. CREATE ONE KEY INTERACTION FEATURE
    print("\n3. Creating key interaction feature...")
    if 'income_group_encoded' in model_df.columns and 'digital_engagement_score' in model_df.columns:
        model_df['income_digital_interaction'] = (
            model_df['income_group_encoded'] * model_df['digital_engagement_score']
        )
        print(f"  ✅ Income-Digital interaction")
        feature_log.append("Income-Digital interaction")
    
    # 4. CREATE BINARY FLAGS FOR HIGH-VALUE SEGMENTS
    print("\n4. Creating strategic binary flags...")
    
    # High financial activity flag
    if 'financial_activity_score' in model_df.columns:
        model_df['high_financial_activity'] = (
            model_df['financial_activity_score'] > model_df['financial_activity_score'].quantile(0.7)
        ).astype(int)
        print(f"  ✅ High financial activity flag")
        feature_log.append("High financial activity flag")
    
    # Digital native flag
    if 'digital_engagement_score' in model_df.columns:
        model_df['digital_native'] = (
            model_df['digital_engagement_score'] > 0.5
        ).astype(int)
        print(f"  ✅ Digital native flag")
        feature_log.append("Digital native flag")
    
    # 5. FINAL OPTIMIZATION - REMOVE LOW-VARIANCE FEATURES IF NEEDED
    print(f"\n5. Final optimization...")
    current_features = model_df.shape[1]
    print(f"Current feature count: {current_features}")
    
    if current_features > max_features:
        print(f"Need to reduce {current_features - max_features} features")
        
        # Calculate variance for numeric columns
        numeric_cols = model_df.select_dtypes(include=['float64', 'int64']).columns
        variances = model_df[numeric_cols].var().sort_values()
        
        # Remove lowest variance features (excluding encoded categoricals and target if present)
        protected_cols = [col for col in model_df.columns if 'encoded' in col or col == 'account']
        low_var_cols = [col for col in variances.head(current_features - max_features).index 
                       if col not in protected_cols]
        
        if low_var_cols:
            model_df = model_df.drop(low_var_cols, axis=1)
            print(f"  ✅ Removed {len(low_var_cols)} low-variance features: {low_var_cols}")
            feature_log.append(f"Removed {len(low_var_cols)} low-variance features")
    
    # FINAL SUMMARY
    print(f"\n📊 FEATURE ENGINEERING SUMMARY")
    print("=" * 50)
    print(f"Original columns: {df.shape[1]}")
    print(f"Final columns: {model_df.shape[1]}")
    print(f"Feature reduction: {df.shape[1] - model_df.shape[1]} columns removed/combined")
    print(f"Target achieved: {'✅' if model_df.shape[1] <= max_features else '❌'}")
    
    print(f"\nTransformations applied:")
    for i, log_entry in enumerate(feature_log, 1):
        print(f"  {i}. {log_entry}")
    
    print(f"\nFinal feature list ({model_df.shape[1]} columns):")
    for i, col in enumerate(sorted(model_df.columns), 1):
        print(f"  {i:2d}. {col}")
    
    return model_df, label_encoders, feature_log

In [9]:

# Usage
df_modell, encoders, feature_log = simplified_feature_engineering(df_fixed, max_features=30)


🔧 SIMPLIFIED FEATURE ENGINEERING
Starting with 30 columns, targeting max 30 columns

1. Encoding essential categorical variables...


  ✅ region → region_encoded (7 categories)
  ✅ income_group → income_group_encoded (4 categories)
  ✅ demo_group → demo_group_encoded (7 categories)

2. Creating composite engagement scores...
  ✅ Digital engagement score from 6 columns
  ✅ Financial activity score from 3 columns
  ✅ Government services score from 3 columns

3. Creating key interaction feature...
  ✅ Income-Digital interaction

4. Creating strategic binary flags...
  ✅ High financial activity flag
  ✅ Digital native flag

5. Final optimization...
Current feature count: 27

📊 FEATURE ENGINEERING SUMMARY
Original columns: 30
Final columns: 27
Feature reduction: 3 columns removed/combined
Target achieved: ✅

Transformations applied:
  1. Encoded region
  2. Encoded income_group
  3. Encoded demo_group
  4. Digital engagement score (combined 6 features)
  5. Financial activity score (from 3 features)
  6. Government services score (combined 3 features)
  7. Income-Digital interaction
  8. High financial activity flag
  9. 

In [10]:
df_modell.shape, df_modell.columns.tolist()

((8311, 27),
 ['demo_subgroup',
  'has_account',
  'borrowed_any',
  'credit_card',
  'biz_loan_source',
  'biz_loan',
  'loan_purpose_group',
  'loan_purpose',
  'saved_old_age',
  'saved_for_purchase',
  'saved_no_purpose',
  'mobile_pay_s_r',
  'fin_resilience',
  'emergency_funds',
  'prefer_digital_acc',
  'prefer_digital_fin',
  'saved_any',
  'region_clean',
  'region_encoded',
  'income_group_encoded',
  'demo_group_encoded',
  'digital_engagement_score',
  'financial_activity_score',
  'govt_services_score',
  'income_digital_interaction',
  'high_financial_activity',
  'digital_native'])

### Final dataset contains:

* Encoded categories
* Composite scores (digital, financial, government)
* Interaction + binary flags
* A few raw core features that remain useful (`saved_old_age`, `credit_card`, etc.)



### Results (Example Run)

* **Starting columns:** 29
* **Final columns:** 26
* **Features removed/combined:** 3
* **Target achieved:** (≤ 30 columns)

#### Transformations Applied

1. Encoded `region`
2. Encoded `income_group`
3. Encoded `demo_group`
4. Digital engagement score (6 features combined)
5. Financial activity score (3 features combined)
6. Government services score (3 features combined)
7. Income–Digital interaction
8. High financial activity flag
9. Digital native flag

---


This pipeline **shrinks features** while keeping important signals about **demographics, digital adoption, financial activity, and government services**. Perfect balance of **compactness + predictive power**.

---


## Machine Learnng Pipeline 

In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score, 
    f1_score, confusion_matrix, matthews_corrcoef, 
    balanced_accuracy_score, log_loss, average_precision_score
)

def simple_ml_pipeline(df, target_col='has_account', exclude_cols=None):
    """
    Simple, reliable ML pipeline with comprehensive analysis
    """
    print("🚀 SIMPLE ML PIPELINE")
    print("="*50)
    
    if exclude_cols is None:
        exclude_cols = []
    
    # 1. PREPARE DATA
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    feature_cols = [col for col in df.columns if col not in 
                   [target_col] + categorical_cols + exclude_cols]
    
    X = df[feature_cols]
    y = (df[target_col] > 0.5).astype(int)
    
    print(f"✅ Features: {len(feature_cols)}")
    print(f"✅ Features used: {feature_cols}")
    print(f"✅ Target distribution: {y.value_counts().to_dict()}")
    
    # Check for missing values
    missing_summary = X.isnull().sum()
    if missing_summary.sum() > 0:
        print(f"⚠️ Missing values found in {(missing_summary > 0).sum()} features")
        print(f"   Total missing: {missing_summary.sum()} ({missing_summary.sum()/len(X)*100:.1f}%)")
    
    # 2. HANDLE MISSING VALUES
    imputer = SimpleImputer(strategy='median')
    X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
    
    # 3. SPLIT DATA
    X_train, X_test, y_train, y_test = train_test_split(
        X_imputed, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"✅ Train: {X_train.shape}, Test: {X_test.shape}")
    
    # 4. SCALE FEATURES
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 5. TRAIN MODELS
    models = {
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
        'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
        'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=100),
        'SVM': SVC(random_state=42, probability=True)
    }
    
    results = []
    
    print("\n📊 MODEL RESULTS:")
    print("-" * 60)
    print(f"{'Model':<20} {'Accuracy':<10} {'AUC':<10}")
    print("-" * 60)
    
    for name, model in models.items():
        # Use scaled data for Logistic Regression and SVM
        if name in ['Logistic Regression', 'SVM']:
            X_train_use, X_test_use = X_train_scaled, X_test_scaled
        else:
            X_train_use, X_test_use = X_train.values, X_test.values
        
        try:
            # Train
            model.fit(X_train_use, y_train)
            
            # Predict
            y_pred = model.predict(X_test_use)
            y_prob = model.predict_proba(X_test_use)[:, 1]
            
            # Calculate metrics
            accuracy = accuracy_score(y_test, y_pred)
            auc = roc_auc_score(y_test, y_prob)
            
            results.append({
                'Model': name,
                'Accuracy': accuracy,
                'AUC': auc,
                'model_obj': model,
                'y_pred': y_pred,
                'y_prob': y_prob
            })
            
            print(f"{name:<20} {accuracy:<10.4f} {auc:<10.4f}")
            
        except Exception as e:
            print(f"{name:<20} ERROR: {str(e)}")
    
    # 6. FIND RANDOM FOREST MODEL
    rf_result = next((r for r in results if r['Model'] == 'Random Forest'), None)
    
    if rf_result:
        print(f"\n🌲 COMPREHENSIVE RANDOM FOREST ANALYSIS")
        print("="*70)
        
        rf_model = rf_result['model_obj']
        rf_pred = rf_result['y_pred']
        rf_prob = rf_result['y_prob']
        
        # Basic Classification Metrics
        print(f"\n📊 CLASSIFICATION METRICS:")
        print("-" * 40)
        
        precision = precision_score(y_test, rf_pred)
        recall = recall_score(y_test, rf_pred)
        f1 = f1_score(y_test, rf_pred)
        balanced_acc = balanced_accuracy_score(y_test, rf_pred)
        mcc = matthews_corrcoef(y_test, rf_pred)
        logloss = log_loss(y_test, rf_prob)
        avg_precision = average_precision_score(y_test, rf_prob)
        
        print(f"Accuracy:              {rf_result['Accuracy']:.4f}")
        print(f"Balanced Accuracy:     {balanced_acc:.4f}")
        print(f"Precision:             {precision:.4f}")
        print(f"Recall (Sensitivity):  {recall:.4f}")
        print(f"F1-Score:              {f1:.4f}")
        print(f"AUC-ROC:               {rf_result['AUC']:.4f}")
        print(f"AUC-PR:                {avg_precision:.4f}")
        print(f"Matthews Corr Coef:    {mcc:.4f}")
        print(f"Log Loss:              {logloss:.4f}")
        
        # Confusion Matrix Analysis
        cm = confusion_matrix(y_test, rf_pred)
        tn, fp, fn, tp = cm.ravel()
        specificity = tn / (tn + fp)
        
        print(f"\n📋 CONFUSION MATRIX ANALYSIS:")
        print("-" * 40)
        print(f"True Negatives:  {tn}")
        print(f"False Positives: {fp}")
        print(f"False Negatives: {fn}")
        print(f"True Positives:  {tp}")
        print(f"Specificity:     {specificity:.4f}")
        
        # Model-Specific Random Forest Metrics
        print(f"\n🌳 RANDOM FOREST SPECIFIC METRICS:")
        print("-" * 40)
        print(f"Number of Trees:       {rf_model.n_estimators}")
        print(f"Max Depth:             {rf_model.max_depth}")
        print(f"Min Samples Split:     {rf_model.min_samples_split}")
        print(f"Min Samples Leaf:      {rf_model.min_samples_leaf}")
        print(f"Max Features:          {rf_model.max_features}")
        print(f"Bootstrap:             {rf_model.bootstrap}")
        
        # Out-of-Bag Score (if bootstrap=True)
        if rf_model.bootstrap and hasattr(rf_model, 'oob_score_'):
            print(f"OOB Score:             {rf_model.oob_score_:.4f}")
        
        # Feature Importance Analysis
        if hasattr(rf_model, 'feature_importances_'):
            print(f"\n📈 FEATURE IMPORTANCE ANALYSIS:")
            print("-" * 50)
            
            importance_df = pd.DataFrame({
                'Feature': feature_cols,
                'Importance': rf_model.feature_importances_
            }).sort_values('Importance', ascending=False)
            
            print(f"Top 15 Most Important Features:")
            for i, (_, row) in enumerate(importance_df.head(15).iterrows(), 1):
                bar_length = int(row['Importance'] * 40)  # Visual bar
                bar = "█" * bar_length
                print(f"{i:2d}. {row['Feature']:<25} {row['Importance']:.4f} {bar}")
            
            # Feature importance statistics
            print(f"\nFeature Importance Statistics:")
            print(f"Mean Importance:       {importance_df['Importance'].mean():.6f}")
            print(f"Std Importance:        {importance_df['Importance'].std():.6f}")
            print(f"Max Importance:        {importance_df['Importance'].max():.4f}")
            print(f"Min Importance:        {importance_df['Importance'].min():.6f}")
            
            # Features with zero importance
            zero_importance = (importance_df['Importance'] == 0).sum()
            print(f"Features with 0 importance: {zero_importance}")
            
            # Return importance for later use
            feature_importance_result = importance_df
        
        # Cross-Validation Performance (quick 3-fold)
        print(f"\n🔄 CROSS-VALIDATION PERFORMANCE:")
        print("-" * 40)
        
        try:
            cv_scores = cross_val_score(rf_model, X_train.values, y_train, cv=3, scoring='roc_auc')
            print(f"CV AUC Scores:         {[f'{score:.4f}' for score in cv_scores]}")
            print(f"CV AUC Mean:           {cv_scores.mean():.4f}")
            print(f"CV AUC Std:            {cv_scores.std():.4f}")
        except Exception as e:
            print(f"CV Error: {str(e)}")
        
        # Prediction Confidence Analysis
        print(f"\n🎯 PREDICTION CONFIDENCE ANALYSIS:")
        print("-" * 40)
        
        confidence_bins = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
        print("Confidence Thresholds vs Accuracy:")
        
        for threshold in confidence_bins:
            high_conf_mask = np.maximum(rf_prob, 1-rf_prob) >= threshold
            if high_conf_mask.sum() > 0:
                high_conf_acc = accuracy_score(y_test[high_conf_mask], rf_pred[high_conf_mask])
                coverage = high_conf_mask.mean()
                print(f"≥{threshold:.1f} confidence: {high_conf_acc:.4f} accuracy, {coverage:.1%} coverage")
        
        # Class Distribution in Predictions
        print(f"\n📊 PREDICTION DISTRIBUTION:")
        print("-" * 40)
        prob_stats = pd.Series(rf_prob).describe()
        print(f"Probability Statistics:")
        for stat, value in prob_stats.items():
            print(f"{stat.capitalize():<10}: {value:.4f}")
    
    # Return enhanced results
    results_df = pd.DataFrame(results)[['Model', 'Accuracy', 'AUC']]
    results_df = results_df.sort_values('AUC', ascending=False)
    
    # Return additional useful objects
    return_dict = {
        'results_df': results_df,
        'best_model': rf_model if rf_result else None,
        'feature_importance': feature_importance_result if rf_result and 'feature_importance_result' in locals() else None,
        'feature_cols': feature_cols,
        'scaler': scaler,
        'imputer': imputer
    }
    
    return return_dict




## Simple ML Pipeline Workflow

#### **Step 1: Data Preparation**

**What's happening:**
- **Identifies categorical columns** (text data like 'region', 'income_group')
- **Creates feature list** by excluding target variable and categorical columns
- **Assumes categorical data is already encoded** or will be dropped

**Why this matters:**
- Most ML algorithms need numeric data only
- Prevents accidental inclusion of target variable in features
- Gives clean separation between features (X) and target (y)

---

#### **Step 2: Target Variable Processing**

**What's happening:**
- **Converts target to binary** (0 or 1)
- **Handles any potential continuous values** by thresholding at 0.5
- **Ensures consistent data type** for classification

**Why this matters:**
- Classification algorithms expect binary targets
- Removes ambiguity about what constitutes "has account"

---

#### **Step 3: Missing Value Imputation**

**What's happening:**
- **Replaces missing values with median** of each column
- **Preserves column names** after transformation
- **Uses median (not mean)** to handle outliers better

---

#### **Step 4: Train-Test Split**

**What's happening:**
- **80% training, 20% testing** split
- **Stratified sampling** maintains same class distribution in both sets
- **Fixed random seed** for reproducible results
- **Prevents data leakage** by separating before any processing

**Why this matters:**
- Gives honest assessment of model performance
- Maintains class balance in both sets
- Ensures reproducible experiments

---

#### **Step 5: Feature Scaling**
**What's happening:**
- **Standardizes features** (mean=0, std=1)
- **Fits scaler only on training data** (prevents data leakage)
- **Applies same transformation to test data**
- **Creates two versions:** scaled and unscaled

**Why this matters:**
- Some algorithms (Logistic Regression, SVM) need scaled features
- Tree-based algorithms work fine with original scale
- Prevents features with large values from dominating

---

#### **Step 6: Model Training & Selection**

**What's happening:**
- **Smart algorithm selection:** Uses scaled data for distance-based algorithms
- **Trains multiple models** with same data splits
- **Calculates consistent metrics** for fair comparison
- **Handles errors gracefully** with try-catch blocks

---

In [12]:
# Usage:
results = simple_ml_pipeline(df_modell, target_col='has_account')

🚀 SIMPLE ML PIPELINE
✅ Features: 24
✅ Features used: ['borrowed_any', 'credit_card', 'biz_loan_source', 'biz_loan', 'loan_purpose_group', 'loan_purpose', 'saved_old_age', 'saved_for_purchase', 'saved_no_purpose', 'mobile_pay_s_r', 'fin_resilience', 'emergency_funds', 'prefer_digital_acc', 'prefer_digital_fin', 'saved_any', 'region_encoded', 'income_group_encoded', 'demo_group_encoded', 'digital_engagement_score', 'financial_activity_score', 'govt_services_score', 'income_digital_interaction', 'high_financial_activity', 'digital_native']
✅ Target distribution: {1: 5141, 0: 3170}
✅ Train: (6648, 24), Test: (1663, 24)

📊 MODEL RESULTS:
------------------------------------------------------------
Model                Accuracy   AUC       
------------------------------------------------------------
Logistic Regression  0.8870     0.9585    
Random Forest        0.9308     0.9825    
Gradient Boosting    0.9176     0.9773    
SVM                  0.9200     0.9745    

🌲 COMPREHENSIVE RANDO


##  **Results Analysis:**

#### **Results:**
- **Random Forest AUC: 0.9824** (98.24% - Almost perfect!)
- **Accuracy: 92.96%** (Extremely high)
- **Precision & Recall both >90%** (Suspiciously balanced)

#### **Red Flags Indicating Data Leakage/Overfitting:**

#### **1. Unrealistic Performance**
- **Real-world financial inclusion models** typically get 70-85% AUC
- **98%+ AUC** suggests the model has "seen the answers"
- **Perfect precision/recall balance** rarely occurs in practice

#### **2. Missing Data Treatment Problems**
- **KNN imputation with 95,209+ missing values** creates artificial patterns
- **Missing financial data ≠ Random missing** - it often means "no activity"
- **Median imputation** assumes missing values follow same distribution

#### **3. Feature Engineering Over-optimization**
- **24 features** might include derived features that leak information
- **Composite scores** might be inadvertently including target information
- **Interactive features** could be creating perfect separators

---



### **The Core Problem: Aggressive Missing Data Imputation**

#### **What Likely Happened:**
1. **Original dataset had meaningful missing patterns**
2. **KNN imputation "filled in the blanks"** with estimated values
3. **These estimates created artificial relationships**
4. **Model learned these artificial patterns instead of real ones**

#### **Why This Creates False Performance:**
- **Missing mobile payment data** likely means "doesn't use mobile payments" (should be 0, not estimated)
- **Missing credit card data** likely means "no credit card" (should be 0, not median)
- **Missing savings data** could mean "no formal savings" (should be 0, not KNN estimate)

---

#### **Next Steps: Better Data Cleaning**

Let's implement **domain-aware cleaning** that:

1. **Respects the meaning of missing financial data**
2. **Creates missingness indicators** to capture patterns
3. **Uses conservative imputation strategies**
4. **Validates results against business logic**

This will give you **more realistic but trustworthy results** that actually reflect financial inclusion patterns rather than imputation artifacts.

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score, 
    f1_score, confusion_matrix, matthews_corrcoef, 
    balanced_accuracy_score, log_loss, average_precision_score
)

def enhanced_ml_pipeline(df, target_col='has_account', exclude_cols=None):
    """
    Fixed Enhanced ML pipeline showing BOTH training and test performance accurately
    """
    print("🚀 ENHANCED ML PIPELINE - TRAIN vs TEST ANALYSIS")
    print("="*60)
    
    if exclude_cols is None:
        exclude_cols = []
    
    # 1. PREPARE DATA
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    feature_cols = [col for col in df.columns if col not in 
                   [target_col] + categorical_cols + exclude_cols]
    
    X = df[feature_cols]
    y = (df[target_col] > 0.5).astype(int)
    
    print(f"✅ Features: {len(feature_cols)}")
    print(f"✅ Target distribution: {y.value_counts().to_dict()}")
    
    # Check for missing values
    missing_summary = X.isnull().sum()
    if missing_summary.sum() > 0:
        print(f"⚠️ Missing values found in {(missing_summary > 0).sum()} features")
        print(f"   Total missing: {missing_summary.sum()} ({missing_summary.sum()/len(X)*100:.1f}%)")
    
    # 2. SPLIT DATA FIRST (before any preprocessing)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"✅ Train: {X_train.shape}, Test: {X_test.shape}")
    
    # 3. HANDLE MISSING VALUES (fit on train, transform both)
    imputer = SimpleImputer(strategy='median')
    X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train), 
                                  columns=X_train.columns, index=X_train.index)
    X_test_imputed = pd.DataFrame(imputer.transform(X_test), 
                                 columns=X_test.columns, index=X_test.index)
    
    # 4. SCALE FEATURES (fit on train, transform both)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imputed)
    X_test_scaled = scaler.transform(X_test_imputed)
    
    # 5. TRAIN MODELS AND EVALUATE ON BOTH SETS
    models = {
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
        'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
        'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=100),
        'SVM': SVC(random_state=42, probability=True)
    }
    
    results = []
    
    print(f"\n📊 MODEL PERFORMANCE COMPARISON:")
    print("="*80)
    print(f"{'Model':<20} {'TRAIN Acc':<12} {'TRAIN AUC':<12} {'TEST Acc':<12} {'TEST AUC':<12} {'Overfit?':<10}")
    print("="*80)
    
    for name, model in models.items():
        # Use scaled data for Logistic Regression and SVM, raw for tree-based models
        if name in ['Logistic Regression', 'SVM']:
            X_train_use, X_test_use = X_train_scaled, X_test_scaled
        else:
            X_train_use, X_test_use = X_train_imputed.values, X_test_imputed.values
        
        try:
            # Train model
            model.fit(X_train_use, y_train)
            
            # TRAINING SET PREDICTIONS
            y_train_pred = model.predict(X_train_use)
            y_train_prob = model.predict_proba(X_train_use)[:, 1]
            train_accuracy = accuracy_score(y_train, y_train_pred)
            train_auc = roc_auc_score(y_train, y_train_prob)
            
            # TEST SET PREDICTIONS
            y_test_pred = model.predict(X_test_use)
            y_test_prob = model.predict_proba(X_test_use)[:, 1]
            test_accuracy = accuracy_score(y_test, y_test_pred)
            test_auc = roc_auc_score(y_test, y_test_prob)
            
            # OVERFITTING DETECTION
            acc_gap = train_accuracy - test_accuracy
            auc_gap = train_auc - test_auc
            overfitting = "Yes" if (acc_gap > 0.05 or auc_gap > 0.05) else "No"
            
            results.append({
                'Model': name,
                'Train_Accuracy': train_accuracy,
                'Train_AUC': train_auc,
                'Test_Accuracy': test_accuracy,
                'Test_AUC': test_auc,
                'Acc_Gap': acc_gap,
                'AUC_Gap': auc_gap,
                'Overfitting': overfitting,
                'model_obj': model,
                'y_train_pred': y_train_pred,
                'y_test_pred': y_test_pred,
                'y_train_prob': y_train_prob,
                'y_test_prob': y_test_prob
            })
            
            print(f"{name:<20} {train_accuracy:<12.4f} {train_auc:<12.4f} {test_accuracy:<12.4f} {test_auc:<12.4f} {overfitting:<10}")
            
        except Exception as e:
            print(f"{name:<20} ERROR: {str(e)}")
    
    # 6. DETAILED RANDOM FOREST ANALYSIS
    rf_result = next((r for r in results if r['Model'] == 'Random Forest'), None)
    
    if rf_result:
        print(f"\n🌲 DETAILED RANDOM FOREST ANALYSIS")
        print("="*70)
        
        rf_model = rf_result['model_obj']
        
        # TRAINING SET METRICS
        print(f"\n📊 TRAINING SET PERFORMANCE:")
        print("-" * 40)
        
        train_precision = precision_score(y_train, rf_result['y_train_pred'])
        train_recall = recall_score(y_train, rf_result['y_train_pred'])
        train_f1 = f1_score(y_train, rf_result['y_train_pred'])
        train_balanced_acc = balanced_accuracy_score(y_train, rf_result['y_train_pred'])
        train_mcc = matthews_corrcoef(y_train, rf_result['y_train_pred'])
        train_logloss = log_loss(y_train, rf_result['y_train_prob'])
        train_avg_precision = average_precision_score(y_train, rf_result['y_train_prob'])
        
        print(f"Accuracy:              {rf_result['Train_Accuracy']:.4f}")
        print(f"Balanced Accuracy:     {train_balanced_acc:.4f}")
        print(f"Precision:             {train_precision:.4f}")
        print(f"Recall:                {train_recall:.4f}")
        print(f"F1-Score:              {train_f1:.4f}")
        print(f"AUC-ROC:               {rf_result['Train_AUC']:.4f}")
        print(f"AUC-PR:                {train_avg_precision:.4f}")
        print(f"Matthews Corr Coef:    {train_mcc:.4f}")
        print(f"Log Loss:              {train_logloss:.4f}")
        
        # TEST SET METRICS
        print(f"\n📊 TEST SET PERFORMANCE:")
        print("-" * 40)
        
        test_precision = precision_score(y_test, rf_result['y_test_pred'])
        test_recall = recall_score(y_test, rf_result['y_test_pred'])
        test_f1 = f1_score(y_test, rf_result['y_test_pred'])
        test_balanced_acc = balanced_accuracy_score(y_test, rf_result['y_test_pred'])
        test_mcc = matthews_corrcoef(y_test, rf_result['y_test_pred'])
        test_logloss = log_loss(y_test, rf_result['y_test_prob'])
        test_avg_precision = average_precision_score(y_test, rf_result['y_test_prob'])
        
        print(f"Accuracy:              {rf_result['Test_Accuracy']:.4f}")
        print(f"Balanced Accuracy:     {test_balanced_acc:.4f}")
        print(f"Precision:             {test_precision:.4f}")
        print(f"Recall:                {test_recall:.4f}")
        print(f"F1-Score:              {test_f1:.4f}")
        print(f"AUC-ROC:               {rf_result['Test_AUC']:.4f}")
        print(f"AUC-PR:                {test_avg_precision:.4f}")
        print(f"Matthews Corr Coef:    {test_mcc:.4f}")
        print(f"Log Loss:              {test_logloss:.4f}")
        
        # OVERFITTING ANALYSIS
        print(f"\n🔍 OVERFITTING ANALYSIS:")
        print("-" * 40)
        print(f"Accuracy Gap:          {rf_result['Acc_Gap']:.4f} ({'GOOD' if abs(rf_result['Acc_Gap']) < 0.05 else 'CONCERN'})")
        print(f"AUC Gap:               {rf_result['AUC_Gap']:.4f} ({'GOOD' if abs(rf_result['AUC_Gap']) < 0.05 else 'CONCERN'})")
        print(f"Precision Gap:         {train_precision - test_precision:.4f}")
        print(f"Recall Gap:            {train_recall - test_recall:.4f}")
        print(f"F1 Gap:                {train_f1 - test_f1:.4f}")
        print(f"Log Loss Gap:          {train_logloss - test_logloss:.4f}")
        
        # CONFUSION MATRICES
        print(f"\n📋 CONFUSION MATRICES:")
        print("-" * 40)
        
        # Training confusion matrix
        train_cm = confusion_matrix(y_train, rf_result['y_train_pred'])
        train_tn, train_fp, train_fn, train_tp = train_cm.ravel()
        train_specificity = train_tn / (train_tn + train_fp) if (train_tn + train_fp) > 0 else 0
        
        print(f"TRAINING SET:")
        print(f"  True Negatives:  {train_tn}")
        print(f"  False Positives: {train_fp}")
        print(f"  False Negatives: {train_fn}")
        print(f"  True Positives:  {train_tp}")
        print(f"  Specificity:     {train_specificity:.4f}")
        
        # Test confusion matrix
        test_cm = confusion_matrix(y_test, rf_result['y_test_pred'])
        test_tn, test_fp, test_fn, test_tp = test_cm.ravel()
        test_specificity = test_tn / (test_tn + test_fp) if (test_tn + test_fp) > 0 else 0
        
        print(f"\nTEST SET:")
        print(f"  True Negatives:  {test_tn}")
        print(f"  False Positives: {test_fp}")
        print(f"  False Negatives: {test_fn}")
        print(f"  True Positives:  {test_tp}")
        print(f"  Specificity:     {test_specificity:.4f}")
        
        # Feature Importance
        if hasattr(rf_model, 'feature_importances_'):
            print(f"\n📈 FEATURE IMPORTANCE ANALYSIS:")
            print("-" * 50)
            
            importance_df = pd.DataFrame({
                'Feature': feature_cols,
                'Importance': rf_model.feature_importances_
            }).sort_values('Importance', ascending=False)
            
            print(f"Top 10 Most Important Features:")
            for i, (_, row) in enumerate(importance_df.head(10).iterrows(), 1):
                bar_length = int(row['Importance'] * 40)  # Visual bar
                bar = "█" * bar_length
                print(f"{i:2d}. {row['Feature']:<25} {row['Importance']:.4f} {bar}")
        
        # Cross-validation on training data to detect overfitting
        print(f"\n🔄 CROSS-VALIDATION ON TRAINING DATA:")
        print("-" * 40)
        try:
            cv_scores = cross_val_score(rf_model, X_train_imputed.values, y_train, cv=5, scoring='roc_auc')
            print(f"CV AUC Scores:         {[f'{score:.4f}' for score in cv_scores]}")
            print(f"CV AUC Mean:           {cv_scores.mean():.4f}")
            print(f"CV AUC Std:            {cv_scores.std():.4f}")
            print(f"Train-CV Gap:          {rf_result['Train_AUC'] - cv_scores.mean():.4f}")
        except Exception as e:
            print(f"CV Error: {str(e)}")
    
    # 7. SUMMARY TABLE
    print(f"\n📊 FINAL SUMMARY TABLE:")
    print("="*80)
    
    summary_df = pd.DataFrame(results)[['Model', 'Train_Accuracy', 'Test_Accuracy', 'Train_AUC', 'Test_AUC', 'Acc_Gap', 'AUC_Gap', 'Overfitting']]
    summary_df = summary_df.sort_values('Test_AUC', ascending=False)
    
    print(f"{'Model':<20} | {'TRAIN ACC':<10} {'TEST ACC':<10} | {'TRAIN AUC':<10} {'TEST AUC':<10} | {'ACC GAP':<8} {'OVERFIT':<8}")
    print("-" * 80)
    for _, row in summary_df.iterrows():
        print(f"{row['Model']:<20} | {row['Train_Accuracy']:<10.4f} {row['Test_Accuracy']:<10.4f} | "
              f"{row['Train_AUC']:<10.4f} {row['Test_AUC']:<10.4f} | {row['Acc_Gap']:<8.4f} {row['Overfitting']:<8}")
    
    return {
        'results_df': summary_df,
        'best_model': rf_model if rf_result else None,
        'feature_cols': feature_cols,
        'scaler': scaler,
        'imputer': imputer,
        'train_test_split': (X_train_imputed, X_test_imputed, y_train, y_test)
    }

In [14]:
# Usage:
results = enhanced_ml_pipeline(df_modell, target_col='has_account')

🚀 ENHANCED ML PIPELINE - TRAIN vs TEST ANALYSIS
✅ Features: 24
✅ Target distribution: {1: 5141, 0: 3170}
✅ Train: (6648, 24), Test: (1663, 24)

📊 MODEL PERFORMANCE COMPARISON:
Model                TRAIN Acc    TRAIN AUC    TEST Acc     TEST AUC     Overfit?  
Logistic Regression  0.8699       0.9492       0.8870       0.9585       No        
Random Forest        1.0000       1.0000       0.9308       0.9825       Yes       
Gradient Boosting    0.9322       0.9837       0.9176       0.9773       No        
SVM                  0.9126       0.9727       0.9200       0.9745       No        

🌲 DETAILED RANDOM FOREST ANALYSIS

📊 TRAINING SET PERFORMANCE:
----------------------------------------
Accuracy:              1.0000
Balanced Accuracy:     1.0000
Precision:             1.0000
Recall:                1.0000
F1-Score:              1.0000
AUC-ROC:               1.0000
AUC-PR:                1.0000
Matthews Corr Coef:    1.0000
Log Loss:              0.0580

📊 TEST SET PERFORMANCE:
----

## REDO

In [15]:
import pandas as pd
import requests

# Fetch the data.
df_clean= pd.read_csv("data_with_missing_values.csv")

df_clean.head()

,regionwb24_hi,incomegroupwb24,group,group2,account_t_d,borrow_any_t_d,fin26a,fin17a_17a1_d,fin17a,fin22a_22a1_22g_d,...,fin32,fin32_acc,fin34a,fin37_38,fin10,fing2p_acc,fing2p_fin,fing2p,save_any_t_d,region_clean
0,South Asia (excluding high income),Low income,all,all,0.090050,NaN,NaN,0.028194,0.028194,NaN,...,NaN,NaN,NaN,NaN,0.008257,NaN,NaN,NaN,NaN,South Asia
1,Europe & Central Asia (excluding high income),Upper middle income,all,all,0.282681,NaN,NaN,0.085616,0.085616,NaN,...,NaN,NaN,NaN,NaN,0.105931,NaN,NaN,NaN,NaN,Europe & Central Asia
2,Middle East & North Africa (excluding high inc...,Lower middle income,all,all,0.332861,NaN,NaN,0.043257,0.043257,NaN,...,NaN,NaN,NaN,NaN,0.011603,NaN,NaN,NaN,NaN,Middle East & North Africa
3,Sub-Saharan Africa (excluding high income),Lower middle income,all,all,0.392035,NaN,NaN,0.159156,0.159156,NaN,...,NaN,NaN,NaN,NaN,0.154860,NaN,NaN,NaN,NaN,Sub-Saharan Africa
4,Latin America & Caribbean (excluding high income),Upper middle income,all,all,0.331302,NaN,NaN,0.038007,0.038007,NaN,...,NaN,NaN,NaN,NaN,0.219409,NaN,NaN,NaN,NaN,Latin America & Caribbean


In [16]:
# Create a summary DataFrame to show data types, missing values, and unique counts
# summary: dtype / missing / unique
summary = pd.DataFrame({
    'dtype': df_clean.dtypes.astype(str),
    'n_missing': df_clean.isnull().sum(),
    'pct_missing': df_clean.isnull().mean(),
    'n_unique': df_clean.nunique(dropna=False)
})
summary = summary.sort_values('pct_missing', ascending=False)
summary.head(20)   # show top 20 by missingness

,dtype,n_missing,pct_missing,n_unique
fin24aSD_ND,float64,5460,0.644172,3016
fin32_n33_34a,float64,5363,0.632728,3109
fing2p_acc,float64,5363,0.632728,3113
fin26a,float64,5307,0.626121,3168
fin31a,float64,5091,0.600637,3383
fin37_38,float64,4943,0.583176,3533
fin32_n33_acc,float64,4941,0.582940,3532
fing2p_fin,float64,4859,0.573266,3616
fin31a_31b,float64,4537,0.535276,3937
fin34a,float64,4531,0.534568,3944




###  Domain-Aware Data Cleaning Steps

#### **Step 1: Understand the Data**

* Map each column to its meaning
* Check % of missing values
---

#### **Step 2: Categorize by Business Logic**

* Group columns into themes (demographics, digital usage, credit, etc.)
* Summarize missingness per group
---

#### **Step 3: Define Missing Data Strategy**

* Decide per feature:

  * **DROP** (too sparse)
  * **FILL (0, MODE, MEDIAN)**
  * **FILL + Missing Indicator**
---

#### **Step 4: Apply Cleaning Strategy**

* Drop sparse columns
* Impute values with rules
* Add missingness indicators where useful
---

#### **Step 5: Validate & Summarize**

* Compare dataset before vs after
* Show missing value reduction
* Log all cleaning actions
* Check target variable distribution
---

After Step 5: Dataset is **clean, consistent, and ready for modeling** 🎯

---



In [17]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

def step1_understand_the_data(df):
    """
    Step 1: Understand what each column represents
    """
    print("🔍 STEP 1: UNDERSTANDING THE DATA")
    print("="*60)
    
    # Based on column names, let's decode what they likely represent
    column_meanings = {
        # Demographics
        'regionwb24_hi': 'World Bank Region (High Income indicator)',
        'incomegroupwb24': 'World Bank Income Group Classification',
        'group': 'Demographic Group (likely age/gender/education)',
        'group2': 'Secondary Demographic Group',
        
        # Core Financial Inclusion Indicators
        'account_t_d': 'Has Account (Traditional/Digital) - TARGET VARIABLE',
        'borrow_any_t_d': 'Borrowed Any Money (Traditional/Digital)',
        'save_any_t_d': 'Saved Any Money (Traditional/Digital)',
        
        # Digital Financial Services (fin prefix = financial)
        'fin26a': 'Digital Payment Usage',
        'fin17a_17a1_d': 'Mobile/Digital Payment Composite',
        'fin17a': 'Mobile Payment Usage',
        'fin22a_22a1_22g_d': 'Digital Payment for Bills/Services',
        'fin22a': 'Bill Payment via Digital',
        
        # Savings Behavior
        'fin24aSD_ND': 'Savings - Semi-formal/Informal (very sparse)',
        'fin24aP': 'Savings - Planned/Purpose',
        'fin24aN': 'Savings - No specific purpose',
        
        # Credit/Borrowing Behavior
        'fin31a_31b': 'Credit/Loan from Financial Institution',
        'fin30': 'Credit/Loan Application',
        'fin31a': 'Formal Credit Usage',
        'fin31d': 'Credit from Family/Friends',
        
        # Government/Institutional Services
        'fin32_n33_34a': 'Government Digital Payment Services',
        'fin32_n33_acc': 'Government Services Account',
        'fin32': 'Government Payment Received',
        'fin32_acc': 'Government Services via Account',
        'fin34a': 'Government Digital Payment',
        
        # Financial Resilience
        'fin37_38': 'Emergency Funds/Financial Resilience',
        'fin10': 'Financial Planning/Future Orientation',
        
        # Digital Preference/Access
        'fing2p_acc': 'Prefer Digital - Account Services',
        'fing2p_fin': 'Prefer Digital - Financial Services',
        'fing2p': 'Prefer Digital - General'
    }
    
    print("COLUMN INTERPRETATION:")
    print("-" * 60)
    for col in df.columns:
        meaning = column_meanings.get(col, "Unknown financial indicator")
        missing_pct = (df[col].isnull().sum() / len(df)) * 100
        print(f"{col:<20} | {missing_pct:>5.1f}% missing | {meaning}")
    
    return column_meanings

def step2_categorize_by_business_logic(df, column_meanings):
    """
    Step 2: Categorize columns by business importance and missing data meaning
    """
    print(f"\n🏷️  STEP 2: CATEGORIZING BY BUSINESS LOGIC")
    print("="*60)
    
    categories = {
        'demographics': ['regionwb24_hi', 'incomegroupwb24', 'group', 'group2'],
        'core_inclusion': ['account_t_d', 'borrow_any_t_d', 'save_any_t_d'],
        'digital_usage': ['fin26a', 'fin17a_17a1_d', 'fin17a', 'fin22a_22a1_22g_d', 'fin22a'],
        'savings_behavior': ['fin24aSD_ND', 'fin24aP', 'fin24aN'],
        'credit_behavior': ['fin31a_31b', 'fin30', 'fin31a', 'fin31d'],
        'govt_services': ['fin32_n33_34a', 'fin32_n33_acc', 'fin32', 'fin32_acc', 'fin34a'],
        'financial_resilience': ['fin37_38', 'fin10'],
        'digital_preference': ['fing2p_acc', 'fing2p_fin', 'fing2p']
    }
    
    print("BUSINESS CATEGORY ANALYSIS:")
    print("-" * 60)
    
    for category, cols in categories.items():
        available_cols = [col for col in cols if col in df.columns]
        if available_cols:
            avg_missing = np.mean([df[col].isnull().mean() for col in available_cols]) * 100
            print(f"{category.upper():<20} | {len(available_cols)} cols | {avg_missing:>5.1f}% avg missing")
            
            for col in available_cols:
                missing_pct = df[col].isnull().mean() * 100
                print(f"  └─ {col:<18} | {missing_pct:>5.1f}% missing")
    
    return categories

def step3_missing_data_strategy(df, categories):
    """
    Step 3: Define domain-specific missing data strategy
    """
    print(f"\n🎯 STEP 3: MISSING DATA STRATEGY")
    print("="*60)
    
    strategy = {}
    
    print("STRATEGY BY CATEGORY:")
    print("-" * 60)
    
    # Demographics: Missing = Unknown, but drop if too sparse
    for col in categories['demographics']:
        if col in df.columns:
            missing_pct = df[col].isnull().mean()
            if missing_pct > 0.3:
                strategy[col] = 'DROP'
                print(f"{col:<20} | DROP (>{missing_pct:.1%} missing)")
            else:
                strategy[col] = 'MODE'
                print(f"{col:<20} | FILL with MODE")
    
    # Core inclusion: These are crucial, missing = 0 (no activity)
    for col in categories['core_inclusion']:
        if col in df.columns:
            strategy[col] = 'ZERO'
            print(f"{col:<20} | FILL with 0 (no activity)")
    
    # Digital usage: Missing = 0 (doesn't use digital)
    for col in categories['digital_usage']:
        if col in df.columns:
            missing_pct = df[col].isnull().mean()
            if missing_pct > 0.6:
                strategy[col] = 'DROP'
                print(f"{col:<20} | DROP (>{missing_pct:.1%} missing)")
            else:
                strategy[col] = 'ZERO_WITH_INDICATOR'
                print(f"{col:<20} | FILL with 0 + create indicator")
    
    # Savings: Missing = 0 (no savings activity)
    for col in categories['savings_behavior']:
        if col in df.columns:
            missing_pct = df[col].isnull().mean()
            if missing_pct > 0.6:
                strategy[col] = 'DROP'
                print(f"{col:<20} | DROP (>{missing_pct:.1%} missing)")
            else:
                strategy[col] = 'ZERO_WITH_INDICATOR'
                print(f"{col:<20} | FILL with 0 + create indicator")
    
    # Credit: Missing = 0 (no credit activity)
    for col in categories['credit_behavior']:
        if col in df.columns:
            missing_pct = df[col].isnull().mean()
            if missing_pct > 0.5:
                strategy[col] = 'DROP'
                print(f"{col:<20} | DROP (>{missing_pct:.1%} missing)")
            else:
                strategy[col] = 'ZERO_WITH_INDICATOR'
                print(f"{col:<20} | FILL with 0 + create indicator")
    
    # Government services: Missing = 0 (doesn't use govt services)
    for col in categories['govt_services']:
        if col in df.columns:
            missing_pct = df[col].isnull().mean()
            if missing_pct > 0.6:
                strategy[col] = 'DROP'
                print(f"{col:<20} | DROP (>{missing_pct:.1%} missing)")
            else:
                strategy[col] = 'ZERO_WITH_INDICATOR'
                print(f"{col:<20} | FILL with 0 + create indicator")
    
    # Financial resilience: Important to keep
    for col in categories['financial_resilience']:
        if col in df.columns:
            missing_pct = df[col].isnull().mean()
            if missing_pct > 0.5:
                strategy[col] = 'DROP'
                print(f"{col:<20} | DROP (>{missing_pct:.1%} missing)")
            else:
                strategy[col] = 'MEDIAN'
                print(f"{col:<20} | FILL with MEDIAN")
    
    # Digital preference: Missing = 0 (no digital preference)
    for col in categories['digital_preference']:
        if col in df.columns:
            missing_pct = df[col].isnull().mean()
            if missing_pct > 0.6:
                strategy[col] = 'DROP'
                print(f"{col:<20} | DROP (>{missing_pct:.1%} missing)")
            else:
                strategy[col] = 'ZERO_WITH_INDICATOR'
                print(f"{col:<20} | FILL with 0 + create indicator")
    
    return strategy

def step4_apply_cleaning_strategy(df, strategy):
    """
    Step 4: Apply the cleaning strategy
    """
    print(f"\n🧹 STEP 4: APPLYING CLEANING STRATEGY")
    print("="*60)
    
    df_clean = df.copy()
    cleaning_log = []
    
    for col, action in strategy.items():
        if col not in df_clean.columns:
            continue
            
        missing_count = df_clean[col].isnull().sum()
        
        if action == 'DROP':
            df_clean.drop(col, axis=1, inplace=True)
            print(f"✂️  DROPPED: {col}")
            cleaning_log.append(f"DROPPED {col} (too sparse)")
            
        elif action == 'ZERO':
            df_clean[col].fillna(0, inplace=True)
            print(f"0️⃣  FILLED with 0: {col} ({missing_count} values)")
            cleaning_log.append(f"FILLED {col} with 0 ({missing_count} missing)")
            
        elif action == 'ZERO_WITH_INDICATOR':
            # Create indicator for missingness
            df_clean[f'{col}_was_missing'] = df_clean[col].isnull().astype(int)
            df_clean[col].fillna(0, inplace=True)
            print(f"0️⃣🏷️  FILLED with 0 + indicator: {col} ({missing_count} values)")
            cleaning_log.append(f"FILLED {col} with 0 + created missing indicator ({missing_count} missing)")
            
        elif action == 'MODE':
            mode_val = df_clean[col].mode()
            fill_val = mode_val.iloc[0] if len(mode_val) > 0 else 'Unknown'
            df_clean[col].fillna(fill_val, inplace=True)
            print(f"📊 FILLED with MODE: {col} = {fill_val} ({missing_count} values)")
            cleaning_log.append(f"FILLED {col} with mode {fill_val} ({missing_count} missing)")
            
        elif action == 'MEDIAN':
            median_val = df_clean[col].median()
            df_clean[col].fillna(median_val, inplace=True)
            print(f"📊 FILLED with MEDIAN: {col} = {median_val} ({missing_count} values)")
            cleaning_log.append(f"FILLED {col} with median {median_val} ({missing_count} missing)")
    
    return df_clean, cleaning_log

def step5_validate_and_summarize(original_df, cleaned_df, cleaning_log):
    """
    Step 5: Validate the cleaning results
    """
    print(f"\n✅ STEP 5: VALIDATION & SUMMARY")
    print("="*60)
    
    print("BEFORE vs AFTER:")
    print(f"  Original shape: {original_df.shape}")
    print(f"  Cleaned shape:  {cleaned_df.shape}")
    print(f"  Columns dropped: {original_df.shape[1] - cleaned_df.shape[1]}")
    print(f"  Rows retained: {cleaned_df.shape[0]/original_df.shape[0]:.1%}")
    
    print(f"\nMISSING VALUES:")
    print(f"  Original: {original_df.isnull().sum().sum():,}")
    print(f"  Cleaned:  {cleaned_df.isnull().sum().sum():,}")
    
    print(f"\nCLEANING ACTIONS TAKEN:")
    for i, action in enumerate(cleaning_log, 1):
        print(f"  {i:2d}. {action}")
    
    # # Check target variable distribution if it exists
    # target_candidates = ['account_t_d']
    # for target in target_candidates:
    #     if target in cleaned_df.columns:
    #         print(f"\nTARGET VARIABLE ({target}) DISTRIBUTION:")
    #         target_dist = cleaned_df[target].value_counts(normalize=True, dropna=False)
    #         for value, pct in target_dist.items():
    #             print(f"  {value}: {pct:.1%}")
    
    
    
    print(f"\n🎯 READY FOR MODELING:")
    print(f"  ✅ No missing values: {cleaned_df.isnull().sum().sum() == 0}")
    print(f"  ✅ Reasonable feature count: {cleaned_df.shape[1]} columns")
    print(f"  ✅ Domain knowledge applied: ✅")
    print(f"  ✅ Overfitting risk reduced: ✅")
    
    return True

def run_complete_cleaning_pipeline(df):
    """
    Run the complete step-by-step cleaning pipeline
    """
    print("🚀 DOMAIN-AWARE FINANCIAL INCLUSION DATA CLEANING")
    print("="*70)
    
    # Step 1: Understand the data
    column_meanings = step1_understand_the_data(df)
    
    # Step 2: Categorize by business logic
    categories = step2_categorize_by_business_logic(df, column_meanings)
    
    # Step 3: Define missing data strategy
    strategy = step3_missing_data_strategy(df, categories)
    
    # Step 4: Apply cleaning strategy
    df_cleaned, cleaning_log = step4_apply_cleaning_strategy(df, strategy)
    
    # Step 5: Validate and summarize
    step5_validate_and_summarize(df, df_cleaned, cleaning_log)
    
    print(f"\n🎉 CLEANING COMPLETE!")
    print(f"   Dataset ready for realistic modeling")
    
    return df_cleaned, cleaning_log, categories

In [18]:

# Usage:
df_cleaned, cleaning_log, categories = run_complete_cleaning_pipeline(df_clean)

🚀 DOMAIN-AWARE FINANCIAL INCLUSION DATA CLEANING
🔍 STEP 1: UNDERSTANDING THE DATA
COLUMN INTERPRETATION:
------------------------------------------------------------
regionwb24_hi        |   8.1% missing | World Bank Region (High Income indicator)
incomegroupwb24      |   8.1% missing | World Bank Income Group Classification
group                |   0.0% missing | Demographic Group (likely age/gender/education)
group2               |   0.0% missing | Secondary Demographic Group
account_t_d          |   0.0% missing | Has Account (Traditional/Digital) - TARGET VARIABLE
borrow_any_t_d       |  25.5% missing | Borrowed Any Money (Traditional/Digital)
fin26a               |  62.6% missing | Digital Payment Usage
fin17a_17a1_d        |  27.0% missing | Mobile/Digital Payment Composite
fin17a               |  32.6% missing | Mobile Payment Usage
fin22a_22a1_22g_d    |  41.5% missing | Digital Payment for Bills/Services
fin22a               |  45.7% missing | Bill Payment via Digital
fin24aSD

In [19]:
df_clean.shape, df_cleaned.shape

((8476, 30), (8476, 36))

In [20]:
df_cleaned.columns.tolist()

['regionwb24_hi',
 'incomegroupwb24',
 'group',
 'group2',
 'account_t_d',
 'borrow_any_t_d',
 'fin17a_17a1_d',
 'fin17a',
 'fin22a_22a1_22g_d',
 'fin22a',
 'fin24aP',
 'fin24aN',
 'fin30',
 'fin31d',
 'fin32_n33_acc',
 'fin32',
 'fin32_acc',
 'fin34a',
 'fing2p_fin',
 'fing2p',
 'save_any_t_d',
 'region_clean',
 'fin17a_17a1_d_was_missing',
 'fin17a_was_missing',
 'fin22a_22a1_22g_d_was_missing',
 'fin22a_was_missing',
 'fin24aP_was_missing',
 'fin24aN_was_missing',
 'fin30_was_missing',
 'fin31d_was_missing',
 'fin32_n33_acc_was_missing',
 'fin32_was_missing',
 'fin32_acc_was_missing',
 'fin34a_was_missing',
 'fing2p_fin_was_missing',
 'fing2p_was_missing']

In [21]:
df_clean.columns.difference(df_cleaned.columns)

Index(['fin10', 'fin24aSD_ND', 'fin26a', 'fin31a', 'fin31a_31b',
       'fin32_n33_34a', 'fin37_38', 'fing2p_acc'],
      dtype='object')

In [22]:
rename_map = {
    'regionwb24_hi': 'region',
    'incomegroupwb24': 'income_group',
    'group': 'demo_group',
    'group2': 'demo_subgroup',
    'account_t_d': 'has_account',
    'borrow_any_t_d': 'borrowed_any',
    'fin26a': 'credit_card',
    'fin17a_17a1_d': 'biz_loan_source',
    'fin17a': 'biz_loan',
    'fin22a_22a1_22g_d': 'loan_purpose_group',
    'fin22a': 'loan_purpose',
    'fin24aSD_ND': 'saved_old_age',
    'fin24aP': 'saved_for_purchase',
    'fin24aN': 'saved_no_purpose',
    'fin31a_31b': 'mobile_pay_s_r',
    'fin30': 'digital_payment_other',
    'fin31a': 'mobile_payment',
    'fin31d': 'mobile_payment_bill',
    'fin32_n33_34a': 'govt_digital_pay',
    'fin32_n33_acc': 'govt_digital_pay_acc',
    'fin32': 'digital_pay',
    'fin32_acc': 'digital_pay_acc',
    'fin34a': 'govt_payment_recv',
    'fin37_38': 'fin_resilience',
    'fin10': 'emergency_funds',
    'fing2p_acc': 'prefer_digital_acc',
    'fing2p_fin': 'prefer_digital_fin',
    'fing2p': 'prefer_digital',
    'save_any_t_d': 'saved_any',
    
    # engineered/extra columns
    'region_clean': 'region_cleaned',
    'fin17a_17a1_d_was_missing': 'biz_loan_source_missing',
    'fin17a_was_missing': 'biz_loan_missing',
    'fin22a_22a1_22g_d_was_missing': 'loan_purpose_group_missing',
    'fin22a_was_missing': 'loan_purpose_missing',
    'fin24aP_was_missing': 'saved_for_purchase_missing',
    'fin24aN_was_missing': 'saved_no_purpose_missing',
    'fin30_was_missing': 'digital_payment_other_missing',
    'fin31d_was_missing': 'mobile_payment_bill_missing',
    'fin32_n33_acc_was_missing': 'govt_digital_pay_acc_missing',
    'fin32_was_missing': 'digital_pay_missing',
    'fin32_acc_was_missing': 'digital_pay_acc_missing',
    'fin34a_was_missing': 'govt_payment_recv_missing',
    'fing2p_fin_was_missing': 'prefer_digital_fin_missing',
    'fing2p_was_missing': 'prefer_digital_missing'
}
df_cleaned.rename(columns=rename_map, inplace=True)

In [23]:
# Create a summary DataFrame to show data types, missing values, and unique counts
# summary: dtype / missing / unique
summary = pd.DataFrame({
    'dtype': df_cleaned.dtypes.astype(str),
    'n_missing': df_cleaned.isnull().sum(),
    'pct_missing': df_cleaned.isnull().mean(),
    'n_unique': df_cleaned.nunique(dropna=False)
})
summary = summary.sort_values('pct_missing', ascending=False)
summary.head(36)   # show top 20 by missingness

,dtype,n_missing,pct_missing,n_unique
region_cleaned,object,684,0.080698,8
region,object,0,0.000000,7
demo_group,object,0,0.000000,7
income_group,object,0,0.000000,4
demo_subgroup,object,0,0.000000,13
has_account,float64,0,0.000000,8288
biz_loan_source,float64,0,0.000000,6189
borrowed_any,float64,0,0.000000,6311
loan_purpose_group,float64,0,0.000000,4952
loan_purpose,float64,0,0.000000,4600


In [24]:
df_MODELING, encoders, feature_log = simplified_feature_engineering(df_cleaned, max_features=30)


🔧 SIMPLIFIED FEATURE ENGINEERING
Starting with 36 columns, targeting max 30 columns

1. Encoding essential categorical variables...
  ✅ region → region_encoded (7 categories)
  ✅ income_group → income_group_encoded (4 categories)
  ✅ demo_group → demo_group_encoded (7 categories)

2. Creating composite engagement scores...
  ✅ Digital engagement score from 5 columns
  ✅ Financial activity score from 2 columns
  ✅ Government services score from 2 columns

3. Creating key interaction feature...
  ✅ Income-Digital interaction

4. Creating strategic binary flags...
  ✅ High financial activity flag
  ✅ Digital native flag

5. Final optimization...
Current feature count: 35
Need to reduce 5 features
  ✅ Removed 5 low-variance features: ['prefer_digital_fin', 'govt_services_score', 'digital_engagement_score', 'loan_purpose_group', 'loan_purpose']

📊 FEATURE ENGINEERING SUMMARY
Original columns: 36
Final columns: 30
Feature reduction: 6 columns removed/combined
Target achieved: ✅

Transformat

In [25]:
results = simple_ml_pipeline(df_MODELING, target_col='has_account')

🚀 SIMPLE ML PIPELINE
✅ Features: 27
✅ Features used: ['borrowed_any', 'biz_loan_source', 'biz_loan', 'saved_for_purchase', 'saved_no_purpose', 'saved_any', 'biz_loan_source_missing', 'biz_loan_missing', 'loan_purpose_group_missing', 'loan_purpose_missing', 'saved_for_purchase_missing', 'saved_no_purpose_missing', 'digital_payment_other_missing', 'mobile_payment_bill_missing', 'govt_digital_pay_acc_missing', 'digital_pay_missing', 'digital_pay_acc_missing', 'govt_payment_recv_missing', 'prefer_digital_fin_missing', 'prefer_digital_missing', 'region_encoded', 'income_group_encoded', 'demo_group_encoded', 'financial_activity_score', 'income_digital_interaction', 'high_financial_activity', 'digital_native']
✅ Target distribution: {1: 5223, 0: 3253}
✅ Train: (6780, 27), Test: (1696, 27)

📊 MODEL RESULTS:
------------------------------------------------------------
Model                Accuracy   AUC       
------------------------------------------------------------
Logistic Regression  0.8

In [26]:
results = enhanced_ml_pipeline(df_MODELING, target_col='has_account')

🚀 ENHANCED ML PIPELINE - TRAIN vs TEST ANALYSIS
✅ Features: 27
✅ Target distribution: {1: 5223, 0: 3253}
✅ Train: (6780, 27), Test: (1696, 27)

📊 MODEL PERFORMANCE COMPARISON:
Model                TRAIN Acc    TRAIN AUC    TEST Acc     TEST AUC     Overfit?  
Logistic Regression  0.8677       0.9354       0.8614       0.9285       No        
Random Forest        0.9869       0.9964       0.9039       0.9637       Yes       
Gradient Boosting    0.9053       0.9705       0.8821       0.9553       No        
SVM                  0.9063       0.9642       0.8797       0.9538       No        

🌲 DETAILED RANDOM FOREST ANALYSIS

📊 TRAINING SET PERFORMANCE:
----------------------------------------
Accuracy:              0.9869
Balanced Accuracy:     0.9864
Precision:             0.9902
Recall:                0.9885
F1-Score:              0.9893
AUC-ROC:               0.9964
AUC-PR:                0.9974
Matthews Corr Coef:    0.9723
Log Loss:              0.0844

📊 TEST SET PERFORMANCE:
----

In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

def strategy_1_conservative_random_forest(X_train, X_test, y_train, y_test):
    """
    Strategy 1: Tune Random Forest to prevent overfitting
    """
    print("🌲 STRATEGY 1: CONSERVATIVE RANDOM FOREST")
    print("="*50)
    
    # Conservative hyperparameters to prevent overfitting
    conservative_rf = RandomForestClassifier(
        n_estimators=50,           # Fewer trees
        max_depth=8,               # Limit depth
        min_samples_split=20,      # Need more samples to split
        min_samples_leaf=10,       # Need more samples per leaf
        max_features=0.3,          # Use fewer features per tree
        bootstrap=True,            # Use bootstrap sampling
        random_state=42
    )
    
    # Train and evaluate
    conservative_rf.fit(X_train, y_train)
    
    train_pred = conservative_rf.predict(X_train)
    test_pred = conservative_rf.predict(X_test)
    train_prob = conservative_rf.predict_proba(X_train)[:, 1]
    test_prob = conservative_rf.predict_proba(X_test)[:, 1]
    
    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)
    train_auc = roc_auc_score(y_train, train_prob)
    test_auc = roc_auc_score(y_test, test_prob)
    
    print(f"Training Accuracy: {train_acc:.4f}")
    print(f"Test Accuracy:     {test_acc:.4f}")
    print(f"Accuracy Gap:      {train_acc - test_acc:.4f}")
    print(f"Training AUC:      {train_auc:.4f}")
    print(f"Test AUC:          {test_auc:.4f}")
    print(f"AUC Gap:           {train_auc - test_auc:.4f}")
    
    return conservative_rf, {
        'train_acc': train_acc, 'test_acc': test_acc,
        'train_auc': train_auc, 'test_auc': test_auc,
        'acc_gap': train_acc - test_acc, 'auc_gap': train_auc - test_auc
    }

def strategy_2_ensemble_with_regularization(X_train, X_test, y_train, y_test):
    """
    Strategy 2: Use multiple models with different regularization approaches
    """
    print(f"\n🎯 STRATEGY 2: ENSEMBLE WITH REGULARIZATION")
    print("="*50)
    
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import GradientBoostingClassifier
    from sklearn.svm import SVC
    
    # Scale features for regularized models
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    models = {
        'Logistic (L2)': LogisticRegression(C=0.1, random_state=42, max_iter=1000),
        'Logistic (L1)': LogisticRegression(C=0.1, penalty='l1', solver='liblinear', random_state=42),
        'Gradient Boost (Low LR)': GradientBoostingClassifier(
            n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42
        ),
        'SVM (RBF)': SVC(C=0.1, gamma='scale', probability=True, random_state=42)
    }
    
    results = {}
    
    for name, model in models.items():
        # Use scaled data for logistic and SVM
        if 'Logistic' in name or 'SVM' in name:
            X_train_use, X_test_use = X_train_scaled, X_test_scaled
        else:
            X_train_use, X_test_use = X_train, X_test
        
        # Train
        model.fit(X_train_use, y_train)
        
        # Evaluate
        train_prob = model.predict_proba(X_train_use)[:, 1]
        test_prob = model.predict_proba(X_test_use)[:, 1]
        
        train_auc = roc_auc_score(y_train, train_prob)
        test_auc = roc_auc_score(y_test, test_prob)
        gap = train_auc - test_auc
        
        results[name] = {
            'train_auc': train_auc,
            'test_auc': test_auc,
            'gap': gap,
            'model': model
        }
        
        print(f"{name:<20} | Train: {train_auc:.4f} | Test: {test_auc:.4f} | Gap: {gap:.4f}")
    
    # Find best model (lowest gap + high test performance)
    best_model = min(results.items(), key=lambda x: x[1]['gap'] + (1 - x[1]['test_auc']))
    print(f"\n🏆 Best Model: {best_model[0]} (Gap: {best_model[1]['gap']:.4f})")
    
    return results

def strategy_3_cross_validation_based_selection(X, y):
    """
    Strategy 3: Use cross-validation to select models that generalize well
    """
    print(f"\n🔄 STRATEGY 3: CROSS-VALIDATION MODEL SELECTION")
    print("="*50)
    
    from sklearn.model_selection import cross_validate
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
    from sklearn.svm import SVC
    
    # Scale for some models
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    models = {
        'Conservative RF': RandomForestClassifier(
            n_estimators=30, max_depth=6, min_samples_leaf=15, random_state=42
        ),
        'Logistic (Regularized)': LogisticRegression(C=0.01, random_state=42, max_iter=1000),
        'Gradient Boost (Conservative)': GradientBoostingClassifier(
            n_estimators=50, learning_rate=0.05, max_depth=3, random_state=42
        )
    }
    
    cv_results = {}
    
    for name, model in models.items():
        # Use scaled data for logistic regression
        X_use = X_scaled if 'Logistic' in name else X
        
        # 5-fold cross-validation
        cv_scores = cross_validate(
            model, X_use, y, 
            cv=5, 
            scoring=['roc_auc', 'accuracy'],
            return_train_score=True
        )
        
        train_auc_mean = cv_scores['train_roc_auc'].mean()
        test_auc_mean = cv_scores['test_roc_auc'].mean()
        train_auc_std = cv_scores['train_roc_auc'].std()
        test_auc_std = cv_scores['test_roc_auc'].std()
        
        gap = train_auc_mean - test_auc_mean
        
        cv_results[name] = {
            'train_auc_mean': train_auc_mean,
            'test_auc_mean': test_auc_mean,
            'train_auc_std': train_auc_std,
            'test_auc_std': test_auc_std,
            'gap': gap
        }
        
        print(f"{name:<25} | CV Train: {train_auc_mean:.4f}±{train_auc_std:.3f}")
        print(f"{'':<25} | CV Test:  {test_auc_mean:.4f}±{test_auc_std:.3f}")
        print(f"{'':<25} | Gap:      {gap:.4f}")
        print("-" * 50)
    
    # Best model has smallest gap + high performance
    best_cv_model = min(cv_results.items(), key=lambda x: x[1]['gap'])
    print(f"🏆 Best CV Model: {best_cv_model[0]} (Gap: {best_cv_model[1]['gap']:.4f})")
    
    return cv_results

def strategy_4_feature_engineering_to_reduce_overfitting(df, target_col):
    """
    Strategy 4: Create robust features that are less prone to overfitting
    """
    print(f"\n🔧 STRATEGY 4: ANTI-OVERFITTING FEATURE ENGINEERING")
    print("="*50)
    
    df_featured = df.copy()
    
    # 1. Create aggregated features (less granular = less overfitting)
    digital_cols = [col for col in df.columns if 'digital' in col.lower() or 'mobile' in col.lower()]
    if digital_cols:
        df_featured['digital_activity_score'] = df_featured[digital_cols].sum(axis=1)
        print(f"✅ Created digital_activity_score from {len(digital_cols)} features")
    
    # 2. Create financial inclusion index
    financial_cols = [col for col in df.columns if any(term in col.lower() 
                     for term in ['save', 'borrow', 'loan', 'credit'])]
    if financial_cols:
        df_featured['financial_inclusion_index'] = df_featured[financial_cols].mean(axis=1)
        print(f"✅ Created financial_inclusion_index from {len(financial_cols)} features")
    
    # 3. Create ratios (more stable than absolute values)
    if 'emergency_funds' in df.columns and 'fin_resilience' in df.columns:
        df_featured['resilience_ratio'] = (df_featured['emergency_funds'] + 0.001) / (df_featured['fin_resilience'] + 0.001)
        print("✅ Created resilience_ratio")
    
    # 4. Binning continuous variables (reduces overfitting to exact values)
    numeric_cols = df_featured.select_dtypes(include=[np.number]).columns
    for col in numeric_cols[:3]:  # Do first 3 for demonstration
        if col != target_col and df_featured[col].nunique() > 10:
            df_featured[f'{col}_binned'] = pd.cut(df_featured[col], bins=5, labels=False)
            print(f"✅ Created {col}_binned (5 bins)")
    
    print(f"\nOriginal features: {df.shape[1]}")
    print(f"Enhanced features: {df_featured.shape[1]}")
    print(f"Added features: {df_featured.shape[1] - df.shape[1]}")
    
    return df_featured

def comprehensive_overfitting_solution(df, target_col):
    """
    Apply all strategies comprehensively
    """
    print("🚀 COMPREHENSIVE OVERFITTING SOLUTION")
    print("="*60)
    
    # Prepare data
    feature_cols = [col for col in df.columns if col != target_col and df[col].dtype in [np.number]]
    X = df[feature_cols]
    y = (df[target_col] > 0.5).astype(int)
    
    # Handle missing values conservatively (simple median, no fancy KNN)
    imputer = SimpleImputer(strategy='median')
    X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X_imputed, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"Data prepared: {X_train.shape[0]} train, {X_test.shape[0]} test samples")
    
    # Apply all strategies
    results = {}
    
    # Strategy 1: Conservative Random Forest
    _, results['conservative_rf'] = strategy_1_conservative_random_forest(
        X_train, X_test, y_train, y_test
    )
    
    # Strategy 2: Ensemble with regularization
    results['ensemble'] = strategy_2_ensemble_with_regularization(
        X_train, X_test, y_train, y_test
    )
    
    # Strategy 3: Cross-validation selection
    results['cv_results'] = strategy_3_cross_validation_based_selection(X_imputed, y)
    
    return results



In [28]:
# Usage example:
df_fit = comprehensive_overfitting_solution(df_MODELING, 'has_account')

🚀 COMPREHENSIVE OVERFITTING SOLUTION
Data prepared: 6780 train, 1696 test samples
🌲 STRATEGY 1: CONSERVATIVE RANDOM FOREST
Training Accuracy: 0.8568
Test Accuracy:     0.8355
Accuracy Gap:      0.0213
Training AUC:      0.9377
Test AUC:          0.9264
AUC Gap:           0.0113

🎯 STRATEGY 2: ENSEMBLE WITH REGULARIZATION
Logistic (L2)        | Train: 0.8391 | Test: 0.8403 | Gap: -0.0013
Logistic (L1)        | Train: 0.8387 | Test: 0.8408 | Gap: -0.0021
Gradient Boost (Low LR) | Train: 0.9377 | Test: 0.9292 | Gap: 0.0085
SVM (RBF)            | Train: 0.8838 | Test: 0.8806 | Gap: 0.0032

🏆 Best Model: Gradient Boost (Low LR) (Gap: 0.0085)

🔄 STRATEGY 3: CROSS-VALIDATION MODEL SELECTION
Conservative RF           | CV Train: 0.9235±0.004
                          | CV Test:  0.9047±0.013
                          | Gap:      0.0188
--------------------------------------------------
Logistic (Regularized)    | CV Train: 0.8395±0.006
                          | CV Test:  0.8334±0.022
       

In [29]:
df_MODELING.columns.tolist()

['demo_subgroup',
 'has_account',
 'borrowed_any',
 'biz_loan_source',
 'biz_loan',
 'saved_for_purchase',
 'saved_no_purpose',
 'saved_any',
 'region_cleaned',
 'biz_loan_source_missing',
 'biz_loan_missing',
 'loan_purpose_group_missing',
 'loan_purpose_missing',
 'saved_for_purchase_missing',
 'saved_no_purpose_missing',
 'digital_payment_other_missing',
 'mobile_payment_bill_missing',
 'govt_digital_pay_acc_missing',
 'digital_pay_missing',
 'digital_pay_acc_missing',
 'govt_payment_recv_missing',
 'prefer_digital_fin_missing',
 'prefer_digital_missing',
 'region_encoded',
 'income_group_encoded',
 'demo_group_encoded',
 'financial_activity_score',
 'income_digital_interaction',
 'high_financial_activity',
 'digital_native']

In [30]:
df_MODELING.shape
df_MODELING.columns.tolist()
df_MODELING.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8476 entries, 0 to 8475
Data columns (total 30 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   demo_subgroup                  8476 non-null   object 
 1   has_account                    8476 non-null   float64
 2   borrowed_any                   8476 non-null   float64
 3   biz_loan_source                8476 non-null   float64
 4   biz_loan                       8476 non-null   float64
 5   saved_for_purchase             8476 non-null   float64
 6   saved_no_purpose               8476 non-null   float64
 7   saved_any                      8476 non-null   float64
 8   region_cleaned                 7792 non-null   object 
 9   biz_loan_source_missing        8476 non-null   int64  
 10  biz_loan_missing               8476 non-null   int64  
 11  loan_purpose_group_missing     8476 non-null   int64  
 12  loan_purpose_missing           8476 non-null   i

## MODEL FOR DEPLOYMENT

In [31]:
 # Run pipeline
results = simple_ml_pipeline(df_MODELING, target_col='has_account')

# ============================================================
# SAVE RANDOM FOREST MODEL
# ============================================================
# Re-train Random Forest on full dataset for final model
X = df_MODELING.drop(columns=['has_account'])
y = (df_MODELING['has_account'] > 0.5).astype(int)

# Use only numeric features
categorical_cols = df_MODELING.select_dtypes(include=['object']).columns.tolist()
exclude_cols = categorical_cols + ['has_account']
feature_cols = [col for col in df_MODELING.columns if col not in exclude_cols]

X = df_MODELING[feature_cols]
y = (df_MODELING['has_account'] > 0.5).astype(int)

imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

rf_final = RandomForestClassifier(
    random_state=42,
    n_estimators=100
)
rf_final.fit(X_imputed, y)

# Save model + preprocessing (pipeline-friendly)
with open("fin_app.pkl", "wb") as f:
    pickle.dump({
        "model": rf_final,
        "imputer": imputer,
        "features": feature_cols
    }, f)

print("✅ Random Forest model saved as 'fin_app.pkl'")


🚀 SIMPLE ML PIPELINE
✅ Features: 27
✅ Features used: ['borrowed_any', 'biz_loan_source', 'biz_loan', 'saved_for_purchase', 'saved_no_purpose', 'saved_any', 'biz_loan_source_missing', 'biz_loan_missing', 'loan_purpose_group_missing', 'loan_purpose_missing', 'saved_for_purchase_missing', 'saved_no_purpose_missing', 'digital_payment_other_missing', 'mobile_payment_bill_missing', 'govt_digital_pay_acc_missing', 'digital_pay_missing', 'digital_pay_acc_missing', 'govt_payment_recv_missing', 'prefer_digital_fin_missing', 'prefer_digital_missing', 'region_encoded', 'income_group_encoded', 'demo_group_encoded', 'financial_activity_score', 'income_digital_interaction', 'high_financial_activity', 'digital_native']
✅ Target distribution: {1: 5223, 0: 3253}
✅ Train: (6780, 27), Test: (1696, 27)

📊 MODEL RESULTS:
------------------------------------------------------------
Model                Accuracy   AUC       
------------------------------------------------------------
Logistic Regression  0.8

##  Model Performance Comparison: Domain-Aware Cleaning Impact

### **Key Performance Changes**

| **Metric** | **Before (KNN Imputed)** | **After (Domain-Aware)** | **Change** | **Interpretation** |
|------------|---------------------------|---------------------------|------------|-------------------|
| **AUC Score** | 0.9824 (98.2%) | 0.9607 (96.1%) | ⬇️ -2.2% | More realistic, less overfitted |
| **Accuracy** | 0.9296 (93.0%) | 0.8962 (89.6%) | ⬇️ -3.4% | Honest performance assessment |
| **Precision (No Account)** | 0.89 | 0.87 | ⬇️ -2% | Slight decrease, still excellent |
| **Recall (No Account)** | 0.94 | 0.85 | ⬇️ -9% | More conservative predictions |
| **Precision (Has Account)** | 0.96 | 0.91 | ⬇️ -5% | More realistic precision |
| **Recall (Has Account)** | 0.93 | 0.92 | ⬇️ -1% | Maintained high recall |

---

## **IMPROVEMENTS EXPLANATION**

### **1. More Realistic Performance Range**
- **Before AUC: 98.2%** → **After AUC: 96.1%**
- **96% AUC is still excellent** but falls within realistic bounds for financial inclusion
- **Real-world financial models** typically achieve 75-90% AUC
- **Suggests the model learned genuine patterns** rather than imputation artifacts

### **2. Reduced Overfitting Indicators**
- **Before: Near-perfect balance** (Precision=0.96, Recall=0.93)
- **After: More natural trade-offs** (Precision=0.91, Recall=0.92)
- **Trade-offs between precision/recall** are normal in real data
- **Less suspicious "too good to be true" performance**

### **3. Better Generalization Capability**
- **Before: Learned from artificial KNN patterns**
- **After: Learned from actual behavioral patterns**
- **Model will likely perform better on new, unseen data**
- **Results are more trustworthy for business decisions**

---


---

## **Business Impact & Interpretability**

### **Top Drivers of Financial Inclusion (Trustworthy Results):**

1. **Business Loan Access (28.1% combined)** 
   - Having business loans/loan sources = strong predictor of account ownership
   - **Business logic**: Entrepreneurs need formal financial services

2. **Emergency Financial Resilience (9.8%)**
   - Having emergency funds = account ownership
   - **Business logic**: Financially resilient people use formal services

3. **Digital Payment Adoption (12.3% combined)**
   - Digital payment usage = account ownership
   - **Business logic**: Digital-savvy users embrace formal banking

### **Actionable Insights for Financial Inclusion:**
- **Target entrepreneurs** and small business owners
- **Promote emergency savings** products
- **Invest in digital payment** infrastructure
- **Focus on financial resilience** education

---


# Account Ownership Prediction Model


### **Model Performance**

Your Random Forest model achieved **exceptional predictive performance** with:
- **96.07% AUC-ROC**: Outstanding ability to distinguish between banked and unbanked individuals
- **97.43% AUC-PR**: Excellent precision-recall balance, crucial for imbalanced datasets
- **89.62% Overall Accuracy**: Strong general performance across all predictions
- **88.83% Balanced Accuracy**: Robust performance accounting for class imbalance

### **Dataset Characteristics & Model Reliability**

#### **Data Quality Indicators**
- **8,476 total observations** with 24 engineered features
- **Class distribution**: 61.6% banked (5,223) vs 38.4% unbanked (3,253)
- **Cross-validation stability**: AUC scores consistently between 94.0-95.4% (std: 0.57%)
- **No zero-importance features**: All 24 variables contribute meaningfully to predictions

#### **Model Confidence Analysis**
Your model demonstrates **excellent calibration**:
- **High-confidence predictions (≥90%)** achieve 98.44% accuracy on 56.8% of cases
- **Medium-confidence predictions (≥70%)** achieve 97.05% acpcuracy on 77.9% of cases
- This suggests the model can **reliably identify both high-risk unbanked individuals and confidently banked poulations**



###  **Key Drivers of Financial Inclusion**

#### **Primary Predictors (Top 5)**
1. **Business Loan Source (16.83% importance)**: Access to business credit infrastructure
2. **Business Loan Access (12.30% importance)**: Entrepreneurial financial needs
3. **Emergency Funds (9.80% importance)**: Financial resilience and savings behavior
4. **Digital Payment Usage (6.36% importance)**: Digital financial ecosystem participation
5. **Digital Payment Account (5.97% importance)**: Digital banking infrastructure access

#### **Secondary Drivers (6th-10th)**
- **Loan Purpose Grouping**: Specific credit needs and financial planning
- **Mobile Payment Send/Receive**: P2P digital transaction behavior
- **Digital Finance Preferences**: Attitudes toward modern banking
- **Government Payment Receipt**: Formal payment system integration
---
---




###  **Strategic Policy Insights**

#### **1. Business-Financial Nexus is Critical**
The dominance of business-related features (29.13% combined importance) suggests that **entrepreneurial financial services are the strongest gateway to broader financial inclusion**. This indicates:
- **SME banking programs** may be more effective than general consumer outreach
- **Business loan accessibility** is a key leverage point for inclusion policies

#### **2. Emergency Preparedness as Financial Inclusion Indicator**
Emergency funds ranking 3rd (9.80% importance) reveals that **financial resilience and account ownership are deeply interconnected**:
- People with emergency savings are more likely to have formal accounts
- **Disaster preparedness programs** could be effective inclusion vehicles

#### **3. Digital Infrastructure is Foundation**
Digital payment features (12.33% combined importance) demonstrate that **digital financial infrastructure is essential**:
- Mobile payment capabilities predict account ownership
- **Digital-first inclusion strategies** are likely to be most effective




##  **Operational Recommendations**

### **High-Impact Targeting Strategy**
Based on your model's confidence analysis:

**Tier 1: Surgical Intervention (28.4% of population)**
- **99.58% accuracy** predictions for highest-risk individuals
- Focus **intensive, personalized outreach** on this group
- Expected **highest conversion rates** with targeted interventions

**Tier 2: Moderate Intervention (28.4% of population)**
- **98.44% accuracy** on moderately at-risk individuals  
- Deploy **scalable, digital-first approaches**
- Balance cost-effectiveness with conversion probability

**Tier 3: Monitoring (43.2% of population)**
- Lower intervention priority but **monitor for status changes**
- Focus on **maintenance and retention** of existing account holders

### **Feature-Based Intervention Design**

**Business Development Track (29.13% model weight)**
- Partner with **microfinance institutions**
- Develop **entrepreneur-focused banking products**
- Create **business banking education programs**

**Digital Readiness Track (12.33% model weight)**
- Invest in **mobile payment infrastructure**
- Develop **digital literacy programs**
- Partner with **mobile network operators**

**Financial Resilience Track (9.80% model weight)**
- Design **emergency savings products**
- Create **disaster preparedness financial programs**
- Integrate **social safety nets with banking**


## DONE


## **Abstract**


This study leverages the **Global Findex 2024 dataset** to address one of the world’s most pressing development challenges: financial exclusion. 
Using a dataset of **8,476 individuals** and 24 engineered features, we developed a **machine learning pipeline** to predict account ownership with high accuracy.

---

Among the models tested, the **Random Forest classifier emerged as the strongest performer**, achieving **89.6% accuracy**, **AUC-ROC of 0.9607**, and **AUC-PR of 0.9743**. Importantly, the model demonstrated excellent calibration, reaching **97% accuracy on medium-confidence predictions** and **98%+ accuracy on high-confidence predictions**, making it highly reliable for targeted outreach strategies.


Feature importance analysis revealed that **business loan access, emergency funds, and digital payment adoption** are the strongest predictors of account ownership. This finding underscores that **entrepreneurship, financial resilience, and digital infrastructure** are critical gateways to inclusion.

---

Beyond technical performance, the project translates predictive power into practice through a **deployed web-based tool** 👉 [**finscopee.streamlit.app**](https://finscopee.streamlit.app/). 

Policymakers and practitioners can now input profile data to receive:

* The **probability of financial exclusion**,
* A **ranked explanation of drivers**, and
* A framework for **tiered, data-driven interventions**.

By blending robust modelling with practical deployment, this work demonstrates how **data science can move financial inclusion from reactive policy to proactive strategy** — ensuring scarce resources reach the people who need them most.


In [35]:
"""
Strategic Financial Inclusion ML Pipeline
Purpose: Production-ready ML pipeline for financial inclusion prediction

This pipeline follows a strategic approach:
1. Fast execution (no extensive grid searches)
2. Clear train/test methodology
3. Business-focused metrics
4. Production-ready components
"""

import pandas as pd
import numpy as np
import json
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score, 
                           precision_score, recall_score, f1_score, 
                           roc_auc_score, classification_report, confusion_matrix)
import lightgbm as lgb
import xgboost as xgb
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

class FinancialInclusionPipeline:
    """
    Strategic ML Pipeline for Financial Inclusion Analysis
    
    Key Design Principles:
    - Fast execution (< 5 minutes total)
    - Clear methodology 
    - Business-focused results
    - Production-ready components
    """
    
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.results = {}
        self.models = {}
        self.scalers = {}
        self.encoders = {}
        
    def step_1_data_preparation(self, df, target_col='Has_Account'):
        """
        STEP 1: DATA PREPARATION
        
        What happens here:
        - Separate features from target
        - Handle categorical variables with label encoding
        - Store preprocessing components for production use
        
        Train/Test Split Strategy:
        - Will be done in step 2 after data prep
        - Using stratified split to maintain class balance
        """
        print("=" * 60)
        print("STEP 1: DATA PREPARATION")
        print("=" * 60)
        
        # Separate features and target
        feature_cols = [col for col in df.columns if col != target_col]
        X = df[feature_cols].copy()
        y = df[target_col].copy()
        
        # Identify variable types
        categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
        numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        
        print(f"Dataset Shape: {df.shape}")
        print(f"Features: {len(feature_cols)}")
        print(f"Categorical: {len(categorical_cols)} {categorical_cols}")
        print(f"Numerical: {len(numerical_cols)}")
        print(f"Target Range: {y.min():.3f} to {y.max():.3f}")
        print(f"Target Mean: {y.mean():.3f}")
        
        # Encode categorical variables
        for col in categorical_cols:
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))
            self.encoders[col] = le
            print(f"Encoded {col}: {len(le.classes_)} categories")
        
        # Store prepared data
        self.X = X
        self.y = y
        self.feature_names = X.columns.tolist()
        self.categorical_cols = categorical_cols
        self.numerical_cols = numerical_cols
        
        print("\n✅ Data preparation completed")
        return X, y
    
    def step_2_train_test_split(self, test_size=0.2):
        """
        STEP 2: TRAIN-TEST SPLIT STRATEGY
        
        What happens here:
        - 80% training, 20% testing (industry standard)
        - Stratified split to maintain target distribution
        - Random state fixed for reproducibility
        - Separate scaling for algorithms that need it
        
        Why this approach:
        - Prevents data leakage (test data never seen during training)
        - Stratified ensures both sets represent the population
        - Proper validation methodology for business deployment
        """
        print("\n" + "=" * 60)
        print("STEP 2: TRAIN-TEST SPLIT METHODOLOGY")
        print("=" * 60)
        
        # Convert to binary for stratified split
        y_binary = (self.y >= 0.5).astype(int)
        
        # Stratified split to maintain class balance
        X_train, X_test, y_train, y_test = train_test_split(
            self.X, self.y, 
            test_size=test_size, 
            random_state=self.random_state,
            stratify=y_binary
        )
        
        y_train_bin, y_test_bin = train_test_split(
            y_binary, test_size=test_size, 
            random_state=self.random_state,
            stratify=y_binary
        )[0], train_test_split(
            y_binary, test_size=test_size, 
            random_state=self.random_state,
            stratify=y_binary
        )[1]
        
        print(f"Training Set: {X_train.shape[0]} samples ({X_train.shape[0]/len(self.X)*100:.1f}%)")
        print(f"Test Set: {X_test.shape[0]} samples ({X_test.shape[0]/len(self.X)*100:.1f}%)")
        print(f"Training Target Mean: {y_train.mean():.3f}")
        print(f"Test Target Mean: {y_test.mean():.3f}")
        print(f"Class Balance Maintained: {abs(y_train.mean() - y_test.mean()) < 0.01}")
        
        # Scale features for algorithms that need it
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Store splits
        self.X_train, self.X_test = X_train, X_test
        self.y_train, self.y_test = y_train, y_test
        self.y_train_bin, self.y_test_bin = y_train_bin, y_test_bin
        self.X_train_scaled, self.X_test_scaled = X_train_scaled, X_test_scaled
        self.scalers['standard'] = scaler
        
        print("\n✅ Train-test split completed")
        return X_train, X_test, y_train, y_test
    
    def step_3_regression_models(self):
        """
        STEP 3: REGRESSION MODELING (Probability Prediction)
        
        What happens here:
        - Train 4 core regression models (fast, production-ready)
        - Predict continuous probability of account ownership
        - Evaluate with R² and RMSE
        - 5-fold cross-validation for stability
        
        Why regression approach:
        - Provides probability scores (more business value)
        - Can convert to classification with optimal thresholds
        - Better for ranking and prioritization
        """
        print("\n" + "=" * 60)
        print("STEP 3: REGRESSION MODELS (Probability Prediction)")
        print("=" * 60)
        
        # Define core models (fast, reliable)
        models = {
            'LightGBM': lgb.LGBMRegressor(
                n_estimators=100, 
                random_state=self.random_state, 
                verbose=-1,
                objective='regression'
            ),
            'XGBoost': xgb.XGBRegressor(
                n_estimators=100, 
                random_state=self.random_state,
                eval_metric='rmse'
            ),
            'Random Forest': RandomForestRegressor(
                n_estimators=100, 
                random_state=self.random_state
            ),
            'Linear Regression': LinearRegression()
        }
        
        regression_results = []
        
        for name, model in models.items():
            print(f"\nTraining {name}...")
            
            # Use scaled data for linear models
            if name == 'Linear Regression':
                X_train_use = self.X_train_scaled
                X_test_use = self.X_test_scaled
            else:
                X_train_use = self.X_train
                X_test_use = self.X_test
            
            # Train model
            model.fit(X_train_use, self.y_train)
            
            # Predictions
            y_pred_train = model.predict(X_train_use)
            y_pred_test = model.predict(X_test_use)
            
            # Clip predictions to valid range [0, 1]
            y_pred_test = np.clip(y_pred_test, 0, 1)
            y_pred_train = np.clip(y_pred_train, 0, 1)
            
            # Metrics
            train_r2 = r2_score(self.y_train, y_pred_train)
            test_r2 = r2_score(self.y_test, y_pred_test)
            test_rmse = np.sqrt(mean_squared_error(self.y_test, y_pred_test))
            
            # Cross-validation
            X_cv = X_train_use
            cv_scores = cross_val_score(model, X_cv, self.y_train, cv=5, scoring='r2')
            
            # Store results
            result = {
                'Model': name,
                'Train_R2': train_r2,
                'Test_R2': test_r2,
                'Test_RMSE': test_rmse,
                'CV_R2_Mean': cv_scores.mean(),
                'CV_R2_Std': cv_scores.std(),
                'Overfitting': train_r2 - test_r2
            }
            
            regression_results.append(result)
            self.models[f'{name}_regression'] = model
            
            print(f"  R² = {test_r2:.4f}, RMSE = {test_rmse:.4f}, CV = {cv_scores.mean():.4f}±{cv_scores.std():.4f}")
        
        # Convert to DataFrame and sort
        reg_df = pd.DataFrame(regression_results).sort_values('Test_R2', ascending=False)
        self.results['regression'] = reg_df
        
        print(f"\n🏆 REGRESSION RESULTS:")
        print(reg_df[['Model', 'Test_R2', 'Test_RMSE', 'CV_R2_Mean', 'Overfitting']].round(4))
        
        return reg_df
    
    def step_4_classification_models(self, probability_threshold=0.5):
        """
        STEP 4: CLASSIFICATION MODELING (Binary Decisions)
        
        What happens here:
        - Convert continuous target to binary classification
        - Train 4 core classification models
        - Evaluate with accuracy, precision, recall, F1, AUC
        - Handle class imbalance properly
        
        Why classification approach:
        - Clear business decisions (account/no account)
        - Standard metrics for stakeholders
        - Complements regression approach
        """
        print("\n" + "=" * 60)
        print(f"STEP 4: CLASSIFICATION MODELS (Threshold: {probability_threshold})")
        print("=" * 60)
        
        print(f"Class Distribution:")
        print(f"  Training: {self.y_train_bin.value_counts().to_dict()}")
        print(f"  Test: {self.y_test_bin.value_counts().to_dict()}")
        
        # Define core models
        models = {
            'LightGBM': lgb.LGBMClassifier(
                n_estimators=100, 
                random_state=self.random_state, 
                verbose=-1,
                objective='binary'
            ),
            'XGBoost': xgb.XGBClassifier(
                n_estimators=100, 
                random_state=self.random_state,
                eval_metric='logloss'
            ),
            'Random Forest': RandomForestClassifier(
                n_estimators=100, 
                random_state=self.random_state
            ),
            'Logistic Regression': LogisticRegression(
                random_state=self.random_state, 
                max_iter=1000
            )
        }
        
        classification_results = []
        
        for name, model in models.items():
            print(f"\nTraining {name}...")
            
            # Use scaled data for logistic regression
            if name == 'Logistic Regression':
                X_train_use = self.X_train_scaled
                X_test_use = self.X_test_scaled
            else:
                X_train_use = self.X_train
                X_test_use = self.X_test
            
            # Train model
            model.fit(X_train_use, self.y_train_bin)
            
            # Predictions
            y_pred_train = model.predict(X_train_use)
            y_pred_test = model.predict(X_test_use)
            y_pred_proba = model.predict_proba(X_test_use)[:, 1]
            
            # Metrics
            train_acc = accuracy_score(self.y_train_bin, y_pred_train)
            test_acc = accuracy_score(self.y_test_bin, y_pred_test)
            precision = precision_score(self.y_test_bin, y_pred_test)
            recall = recall_score(self.y_test_bin, y_pred_test)
            f1 = f1_score(self.y_test_bin, y_pred_test)
            auc = roc_auc_score(self.y_test_bin, y_pred_proba)
            
            # Cross-validation
            cv_scores = cross_val_score(model, X_train_use, self.y_train_bin, cv=5, scoring='accuracy')
            
            # Store results
            result = {
                'Model': name,
                'Train_Accuracy': train_acc,
                'Test_Accuracy': test_acc,
                'Precision': precision,
                'Recall': recall,
                'F1_Score': f1,
                'ROC_AUC': auc,
                'CV_Accuracy_Mean': cv_scores.mean(),
                'CV_Accuracy_Std': cv_scores.std(),
                'Overfitting': train_acc - test_acc
            }
            
            classification_results.append(result)
            self.models[f'{name}_classification'] = model
            
            print(f"  Acc = {test_acc:.4f}, F1 = {f1:.4f}, AUC = {auc:.4f}, CV = {cv_scores.mean():.4f}±{cv_scores.std():.4f}")
        
        # Convert to DataFrame and sort
        clf_df = pd.DataFrame(classification_results).sort_values('Test_Accuracy', ascending=False)
        self.results['classification'] = clf_df
        
        print(f"\n🏆 CLASSIFICATION RESULTS:")
        print(clf_df[['Model', 'Test_Accuracy', 'F1_Score', 'ROC_AUC', 'Overfitting']].round(4))
        
        return clf_df
    
    def step_5_feature_importance(self):
        """
        STEP 5: FEATURE IMPORTANCE ANALYSIS
        
        What happens here:
        - Extract feature importance from best tree-based model
        - Calculate correlation with target
        - Identify top predictors for business focus
        
        Why this matters:
        - Guides intervention strategies
        - Reduces model complexity
        - Provides explainable insights for stakeholders
        """
        print("\n" + "=" * 60)
        print("STEP 5: FEATURE IMPORTANCE ANALYSIS")
        print("=" * 60)
        
        # Get best regression model
        best_reg_model_name = self.results['regression'].iloc[0]['Model']
        best_model = self.models[f'{best_reg_model_name}_regression']
        
        print(f"Using {best_reg_model_name} for feature importance")
        
        # Feature importance (if available)
        feature_importance = None
        if hasattr(best_model, 'feature_importances_'):
            importance_scores = best_model.feature_importances_
            feature_importance = pd.DataFrame({
                'Feature': self.feature_names,
                'Importance': importance_scores
            }).sort_values('Importance', ascending=False)
            
            print(f"\n🎯 TOP 10 MOST IMPORTANT FEATURES:")
            for i, (_, row) in enumerate(feature_importance.head(10).iterrows(), 1):
                print(f"  {i:2d}. {row['Feature']:<30} {row['Importance']:.4f}")
        
        # Correlation analysis
        correlations = self.X.corrwith(self.y).abs().sort_values(ascending=False)
        correlation_df = pd.DataFrame({
            'Feature': correlations.index,
            'Correlation': correlations.values
        })
        
        print(f"\n📊 TOP 10 FEATURES BY CORRELATION:")
        for i, (_, row) in enumerate(correlation_df.head(10).iterrows(), 1):
            print(f"  {i:2d}. {row['Feature']:<30} {row['Correlation']:.4f}")
        
        self.results['feature_importance'] = feature_importance
        self.results['feature_correlation'] = correlation_df
        
        return feature_importance, correlation_df
    
    def step_6_business_insights(self):
        """
        STEP 6: BUSINESS INSIGHTS & DEPLOYMENT STRATEGY
        
        What happens here:
        - Convert model results to business recommendations
        - Calculate ROI projections
        - Define deployment approach
        
        This is where ML becomes business value
        """
        print("\n" + "=" * 60)
        print("STEP 6: BUSINESS INSIGHTS & DEPLOYMENT STRATEGY")
        print("=" * 60)
        
        # Get best models
        best_reg = self.results['regression'].iloc[0]
        best_clf = self.results['classification'].iloc[0]
        
        print(f"🥇 RECOMMENDED MODELS:")
        print(f"  Primary (Probability): {best_reg['Model']} (R² = {best_reg['Test_R2']:.4f})")
        print(f"  Secondary (Binary): {best_clf['Model']} (Accuracy = {best_clf['Test_Accuracy']:.4f})")
        
        # Business impact calculation
        baseline_accuracy = max(self.y_test_bin.mean(), 1 - self.y_test_bin.mean())  # Random guess
        model_accuracy = best_clf['Test_Accuracy']
        improvement = (model_accuracy - baseline_accuracy) / baseline_accuracy
        
        print(f"\n💰 BUSINESS IMPACT ESTIMATION:")
        print(f"  Baseline (random): {baseline_accuracy:.1%}")
        print(f"  Model accuracy: {model_accuracy:.1%}")
        print(f"  Improvement: {improvement:.1%}")
        print(f"  Potential cost savings: {improvement * 50:.0f}% (estimated)")
        
        # Risk scoring framework
        best_reg_model = self.models[f"{best_reg['Model']}_regression"]
        
        # Use appropriate data format
        if best_reg['Model'] == 'Linear Regression':
            X_test_use = self.X_test_scaled
        else:
            X_test_use = self.X_test
            
        risk_scores = 1 - best_reg_model.predict(X_test_use)  # Higher score = higher exclusion risk
        risk_scores = np.clip(risk_scores, 0, 1)
        
        # Risk categories
        high_risk = (risk_scores >= 0.7).sum()
        medium_risk = ((risk_scores >= 0.3) & (risk_scores < 0.7)).sum()
        low_risk = (risk_scores < 0.3).sum()
        
        print(f"\n🎯 POPULATION RISK SEGMENTATION:")
        print(f"  High Risk (≥70%): {high_risk:,} individuals ({high_risk/len(risk_scores):.1%})")
        print(f"  Medium Risk (30-70%): {medium_risk:,} individuals ({medium_risk/len(risk_scores):.1%})")
        print(f"  Low Risk (<30%): {low_risk:,} individuals ({low_risk/len(risk_scores):.1%})")
        
        # Deployment recommendations
        print(f"\n🚀 DEPLOYMENT RECOMMENDATIONS:")
        print(f"  1. Deploy {best_reg['Model']} for probability scoring")
        print(f"  2. Use risk segmentation for resource allocation")
        print(f"  3. Focus interventions on high-risk segment first")
        print(f"  4. Monitor model performance monthly")
        print(f"  5. Retrain quarterly with new data")
        
        # Store business insights
        self.results['business_insights'] = {
            'best_regression': best_reg['Model'],
            'best_classification': best_clf['Model'],
            'accuracy_improvement': improvement,
            'risk_segmentation': {
                'high_risk': high_risk,
                'medium_risk': medium_risk,
                'low_risk': low_risk
            }
        }
        
        return self.results['business_insights']
    
    def step_7_quick_clustering(self, n_clusters=3):
        """
        STEP 7: QUICK CUSTOMER SEGMENTATION
        
        Simple K-means clustering for customer segments
        Fast execution, business-focused insights
        """
        print("\n" + "=" * 60)
        print(f"STEP 7: CUSTOMER SEGMENTATION (K={n_clusters})")
        print("=" * 60)
        
        # Quick clustering on scaled data
        kmeans = KMeans(n_clusters=n_clusters, random_state=self.random_state)
        cluster_labels = kmeans.fit_predict(self.X_train_scaled)
        
        # Analyze clusters
        cluster_analysis = pd.DataFrame({
            'Cluster': cluster_labels,
            'Has_Account': self.y_train
        })
        
        print(f"📊 CLUSTER ANALYSIS:")
        for cluster in range(n_clusters):
            cluster_data = cluster_analysis[cluster_analysis['Cluster'] == cluster]
            size = len(cluster_data)
            account_rate = cluster_data['Has_Account'].mean()
            print(f"  Cluster {cluster}: {size:,} individuals ({size/len(cluster_analysis):.1%})")
            print(f"               Account Rate: {account_rate:.3f}")
        
        self.models['clustering'] = kmeans
        self.results['clustering'] = cluster_analysis
        
        return cluster_analysis
    
    def run_complete_pipeline(self, df, target_col='Has_Account'):
        """
        Execute the complete strategic pipeline
        Fast, focused, production-ready
        """
        print("🚀 STRATEGIC FINANCIAL INCLUSION ML PIPELINE")
        print("=" * 80)
        print("Designed for: Production deployment, Business insights, Fast execution")
        print("=" * 80)
        
        # Execute all steps
        self.step_1_data_preparation(df, target_col)
        self.step_2_train_test_split()
        self.step_3_regression_models()
        self.step_4_classification_models()
        self.step_5_feature_importance()
        self.step_6_business_insights()
        self.step_7_quick_clustering()
        
        print("\n" + "=" * 80)
        print("✅ PIPELINE COMPLETED SUCCESSFULLY!")
        print("=" * 80)
        print("Next steps:")
        print("1. Review business insights above")
        print("2. Use pipeline.predict_new_individual() for new predictions")
        print("3. Deploy best model to production")
        print("=" * 80)
        
        return self.results
    
    def predict_new_individual(self, individual_data):
        """
        Make predictions for new individuals
        Production-ready prediction function
        """
        # Get best model
        best_model_name = self.results['regression'].iloc[0]['Model']
        best_model = self.models[f'{best_model_name}_regression']
        
        # Prepare data (encode if needed)
        X_new = individual_data.copy()
        for col in self.categorical_cols:
            if col in X_new.columns and col in self.encoders:
                try:
                    X_new[col] = self.encoders[col].transform(X_new[col].astype(str))
                except ValueError:
                    # Handle unseen categories
                    X_new[col] = 0
        
        # Scale if needed
        if best_model_name == 'Linear Regression':
            X_new_scaled = self.scalers['standard'].transform(X_new)
            prediction = best_model.predict(X_new_scaled)[0]
        else:
            prediction = best_model.predict(X_new)[0]
        
        # Convert to risk score and recommendation
        risk_score = 1 - prediction
        
        if risk_score >= 0.7:
            risk_level = "HIGH"
            recommendation = "Priority intervention needed"
        elif risk_score >= 0.3:
            risk_level = "MEDIUM"
            recommendation = "Consider targeted programs"
        else:
            risk_level = "LOW"
            recommendation = "Monitor and maintain engagement"
        
        return {
            'account_probability': float(prediction),
            'exclusion_risk_score': float(risk_score),
            'risk_level': risk_level,
            'recommendation': recommendation
        }
        
        
        
        
    def save_best_models(self, directory="saved_models"):
        """
        Save the best regression and classification models to .pkl files
        for production deployment.
        """
        if not os.path.exists(directory):
            os.makedirs(directory)

        # Best regression model
        best_reg_name = self.results['regression'].iloc[0]['Model']
        best_reg_model = self.models[f"{best_reg_name}_regression"]
        with open(os.path.join(directory, f"{best_reg_name}_regression.pkl"), "wb") as f:
            pickle.dump(best_reg_model, f)

        # Best classification model
        best_clf_name = self.results['classification'].iloc[0]['Model']
        best_clf_model = self.models[f"{best_clf_name}_classification"]
        with open(os.path.join(directory, f"{best_clf_name}_classification.pkl"), "wb") as f:
            pickle.dump(best_clf_model, f)

        # Optional: save encoders and scaler for production preprocessing
        with open(os.path.join(directory, "preprocessing.pkl"), "wb") as f:
            pickle.dump({
                
                "encoders": self.encoders,
                "scalers": self.scalers,
                "feature_names": self.feature_names
            }, f)

        print(f"\n💾 Models saved to: {os.path.abspath(directory)}")
        
        
      

In [36]:
# # Run the complete pipeline directly on your dataset
results = pipeline.run_complete_pipeline(df_MODELING, target_col='has_account')


NameError: name 'pipeline' is not defined

In [37]:
# Check that results are populated
print(results['regression'].head())      # <-- Should show model names & scores
print(results['classification'].head())

# Now save
pipeline.save_best_models("saved_models")


KeyError: 'regression'

In [41]:
# Complete ML Pipeline for Financial Inclusion Prediction
# Author: Claude
# Dataset: Financial inclusion data with 31 features, 8566 samples

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR, SVC
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.metrics import (mean_squared_error, r2_score, mean_absolute_error, 
                            accuracy_score, precision_score, recall_score, f1_score, 
                            classification_report, confusion_matrix, roc_auc_score, roc_curve)
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
import xgboost as xgb
import lightgbm as lgb
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Assuming your dataframe is named df_modeling
df_modeling = df_MODELING

print("🚀 COMPREHENSIVE ML PIPELINE FOR FINANCIAL INCLUSION PREDICTION")
print("="*70)

# ============================================================================
# SECTION 1: DATA PREPARATION AND EXPLORATION
# ============================================================================

def prepare_data(df):
    """Prepare data for ML modeling"""
    print("\n📊 DATA PREPARATION")
    print("-" * 30)
    
    # Separate features and target
    target_col = 'has_account'
    feature_cols = [col for col in df.columns if col != target_col]
    
    X = df[feature_cols].copy()
    y = df[target_col].copy()
    
    # Handle categorical variables
    categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
    numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    
    print(f"✅ Features: {len(feature_cols)}")
    print(f"✅ Categorical features: {len(categorical_cols)} - {categorical_cols}")
    print(f"✅ Numerical features: {len(numerical_cols)}")
    print(f"✅ Target range: {y.min():.3f} to {y.max():.3f}")
    
    # Encode categorical variables
    le_dict = {}
    for col in categorical_cols:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])
        le_dict[col] = le
    
    return X, y, categorical_cols, numerical_cols, le_dict

# ============================================================================
# SECTION 2: REGRESSION MODELS (Predicting Continuous Account Probability)
# ============================================================================

def evaluate_regression_models(X, y):
    """Evaluate multiple regression models"""
    print("\n🎯 REGRESSION MODELS EVALUATION")
    print("-" * 40)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Scale features for algorithms that need it
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Define models
    models = {
        'Linear Regression': LinearRegression(),
        'Ridge Regression': Ridge(alpha=1.0),
        'Lasso Regression': Lasso(alpha=0.01),
        'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5),
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
        'XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=42, eval_metric='rmse'),
        'LightGBM': lgb.LGBMRegressor(n_estimators=100, random_state=42, verbose=-1),
        'SVR': SVR(kernel='rbf', C=1.0),
        'KNN': KNeighborsRegressor(n_neighbors=5),
        'Neural Network': MLPRegressor(hidden_layer_sizes=(100, 50), random_state=42, max_iter=500),
        'Decision Tree': DecisionTreeRegressor(random_state=42)
    }
    
    results = []
    
    for name, model in models.items():
        print(f"\n🔄 Training {name}...")
        
        # Use scaled data for models that need it
        if name in ['SVR', 'Neural Network', 'KNN']:
            X_train_model = X_train_scaled
            X_test_model = X_test_scaled
        else:
            X_train_model = X_train
            X_test_model = X_test
        
        try:
            # Train model
            model.fit(X_train_model, y_train)
            
            # Predictions
            y_pred_train = model.predict(X_train_model)
            y_pred_test = model.predict(X_test_model)
            
            # Metrics
            train_r2 = r2_score(y_train, y_pred_train)
            test_r2 = r2_score(y_test, y_pred_test)
            train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
            test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
            test_mae = mean_absolute_error(y_test, y_pred_test)
            
            # Cross-validation
            cv_scores = cross_val_score(model, X_train_model, y_train, cv=5, scoring='r2')
            
            results.append({
                'Model': name,
                'Train_R2': train_r2,
                'Test_R2': test_r2,
                'Train_RMSE': train_rmse,
                'Test_RMSE': test_rmse,
                'Test_MAE': test_mae,
                'CV_R2_Mean': cv_scores.mean(),
                'CV_R2_Std': cv_scores.std(),
                'Overfitting': train_r2 - test_r2
            })
            
            print(f"✅ {name}: Test R² = {test_r2:.4f}, RMSE = {test_rmse:.4f}")
            
        except Exception as e:
            print(f"❌ {name}: Error - {str(e)}")
    
    return pd.DataFrame(results).sort_values('Test_R2', ascending=False), scaler

# ============================================================================
# SECTION 3: CLASSIFICATION MODELS (Convert to Binary Classification)
# ============================================================================

def evaluate_classification_models(X, y, threshold=0.5):
    """Convert regression to classification and evaluate models"""
    print(f"\n🎯 CLASSIFICATION MODELS EVALUATION (Threshold: {threshold})")
    print("-" * 50)
    
    # Convert continuous target to binary
    y_binary = (y >= threshold).astype(int)
    print(f"✅ Class distribution: {y_binary.value_counts().to_dict()}")
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y_binary, test_size=0.2, 
                                                        random_state=42, stratify=y_binary)
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Define models
    models = {
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
        'XGBoost': xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'),
        'LightGBM': lgb.LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
        'SVM': SVC(kernel='rbf', C=1.0, probability=True, random_state=42),
        'KNN': KNeighborsClassifier(n_neighbors=5),
        'Neural Network': MLPClassifier(hidden_layer_sizes=(100, 50), random_state=42, max_iter=500),
        'Decision Tree': DecisionTreeClassifier(random_state=42)
    }
    
    results = []
    
    for name, model in models.items():
        print(f"\n🔄 Training {name}...")
        
        # Use scaled data for models that need it
        if name in ['Logistic Regression', 'SVM', 'Neural Network', 'KNN']:
            X_train_model = X_train_scaled
            X_test_model = X_test_scaled
        else:
            X_train_model = X_train
            X_test_model = X_test
        
        try:
            # Train model
            model.fit(X_train_model, y_train)
            
            # Predictions
            y_pred_train = model.predict(X_train_model)
            y_pred_test = model.predict(X_test_model)
            y_pred_proba = model.predict_proba(X_test_model)[:, 1] if hasattr(model, 'predict_proba') else None
            
            # Metrics
            train_acc = accuracy_score(y_train, y_pred_train)
            test_acc = accuracy_score(y_test, y_pred_test)
            precision = precision_score(y_test, y_pred_test)
            recall = recall_score(y_test, y_pred_test)
            f1 = f1_score(y_test, y_pred_test)
            auc = roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else None
            
            # Cross-validation
            cv_scores = cross_val_score(model, X_train_model, y_train, cv=5, scoring='accuracy')
            
            results.append({
                'Model': name,
                'Train_Accuracy': train_acc,
                'Test_Accuracy': test_acc,
                'Precision': precision,
                'Recall': recall,
                'F1_Score': f1,
                'ROC_AUC': auc,
                'CV_Accuracy_Mean': cv_scores.mean(),
                'CV_Accuracy_Std': cv_scores.std(),
                'Overfitting': train_acc - test_acc
            })
            
            print(f"✅ {name}: Accuracy = {test_acc:.4f}, F1 = {f1:.4f}, AUC = {auc:.4f if auc else 'N/A'}")
            
        except Exception as e:
            print(f"❌ {name}: Error - {str(e)}")
    
    return pd.DataFrame(results).sort_values('Test_Accuracy', ascending=False)

# ============================================================================
# SECTION 4: UNSUPERVISED LEARNING (CLUSTERING & SEGMENTATION)
# ============================================================================

def perform_clustering_analysis(X, y):
    """Perform clustering analysis for customer segmentation"""
    print("\n🔍 UNSUPERVISED LEARNING - CLUSTERING ANALYSIS")
    print("-" * 50)
    
    # Scale data for clustering
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # 1. K-Means Clustering
    print("\n🎯 K-Means Clustering")
    inertias = []
    silhouette_scores = []
    K_range = range(2, 11)
    
    for k in K_range:
        kmeans = KMeans(n_clusters=k, random_state=42)
        labels = kmeans.fit_predict(X_scaled)
        inertias.append(kmeans.inertia_)
        
        from sklearn.metrics import silhouette_score
        sil_score = silhouette_score(X_scaled, labels)
        silhouette_scores.append(sil_score)
        print(f"K={k}: Inertia={kmeans.inertia_:.2f}, Silhouette={sil_score:.3f}")
    
    # Find optimal K
    optimal_k = K_range[np.argmax(silhouette_scores)]
    print(f"✅ Optimal K (by silhouette): {optimal_k}")
    
    # Final K-Means
    final_kmeans = KMeans(n_clusters=optimal_k, random_state=42)
    cluster_labels = final_kmeans.fit_predict(X_scaled)
    
    # Analyze clusters
    cluster_analysis = pd.DataFrame(X)
    cluster_analysis['Cluster'] = cluster_labels
    cluster_analysis['Has_Account'] = y
    
    print(f"\n📊 Cluster Analysis (K={optimal_k}):")
    for cluster in range(optimal_k):
        cluster_data = cluster_analysis[cluster_analysis['Cluster'] == cluster]
        avg_account = cluster_data['Has_Account'].mean()
        size = len(cluster_data)
        print(f"Cluster {cluster}: Size={size} ({size/len(X)*100:.1f}%), Avg_Account={avg_account:.3f}")
    
    # 2. DBSCAN
    print(f"\n🎯 DBSCAN Clustering")
    from sklearn.metrics import silhouette_score
    
    eps_values = [0.3, 0.5, 0.7, 1.0, 1.5]
    best_eps = 0.5
    best_score = -1
    
    for eps in eps_values:
        dbscan = DBSCAN(eps=eps, min_samples=5)
        labels = dbscan.fit_predict(X_scaled)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)
        
        if n_clusters > 1:
            score = silhouette_score(X_scaled, labels)
            print(f"eps={eps}: Clusters={n_clusters}, Noise={n_noise}, Silhouette={score:.3f}")
            if score > best_score:
                best_score = score
                best_eps = eps
    
    # Final DBSCAN
    final_dbscan = DBSCAN(eps=best_eps, min_samples=5)
    dbscan_labels = final_dbscan.fit_predict(X_scaled)
    
    return {
        'kmeans_model': final_kmeans,
        'kmeans_labels': cluster_labels,
        'dbscan_model': final_dbscan,
        'dbscan_labels': dbscan_labels,
        'scaler': scaler,
        'cluster_analysis': cluster_analysis
    }

# ============================================================================
# SECTION 5: ADVANCED MODELS & HYPERPARAMETER TUNING
# ============================================================================

def hyperparameter_tuning(X, y, model_type='regression'):
    """Perform hyperparameter tuning for best models"""
    print(f"\n🔧 HYPERPARAMETER TUNING - {model_type.upper()}")
    print("-" * 50)
    
    if model_type == 'regression':
        y_target = y
        scoring = 'r2'
    else:
        y_target = (y >= 0.5).astype(int)
        scoring = 'accuracy'
    
    X_train, X_test, y_train, y_test = train_test_split(X, y_target, test_size=0.2, random_state=42)
    
    # Define models and parameter grids
    if model_type == 'regression':
        models_params = {
            'Random Forest': {
                'model': RandomForestRegressor(random_state=42),
                'params': {
                    'n_estimators': [100, 200],
                    'max_depth': [10, 20, None],
                    'min_samples_split': [2, 5],
                    'min_samples_leaf': [1, 2]
                }
            },
            'XGBoost': {
                'model': xgb.XGBRegressor(random_state=42, eval_metric='rmse'),
                'params': {
                    'n_estimators': [100, 200],
                    'max_depth': [3, 6, 9],
                    'learning_rate': [0.01, 0.1, 0.2],
                    'subsample': [0.8, 1.0]
                }
            },
            'LightGBM': {
                'model': lgb.LGBMRegressor(random_state=42, verbose=-1),
                'params': {
                    'n_estimators': [100, 200],
                    'max_depth': [5, 10, -1],
                    'learning_rate': [0.01, 0.1, 0.2],
                    'num_leaves': [31, 50, 100]
                }
            }
        }
    else:
        models_params = {
            'Random Forest': {
                'model': RandomForestClassifier(random_state=42),
                'params': {
                    'n_estimators': [100, 200],
                    'max_depth': [10, 20, None],
                    'min_samples_split': [2, 5],
                    'min_samples_leaf': [1, 2]
                }
            },
            'XGBoost': {
                'model': xgb.XGBClassifier(random_state=42, eval_metric='logloss'),
                'params': {
                    'n_estimators': [100, 200],
                    'max_depth': [3, 6, 9],
                    'learning_rate': [0.01, 0.1, 0.2],
                    'subsample': [0.8, 1.0]
                }
            },
            'LightGBM': {
                'model': lgb.LGBMClassifier(random_state=42, verbose=-1),
                'params': {
                    'n_estimators': [100, 200],
                    'max_depth': [5, 10, -1],
                    'learning_rate': [0.01, 0.1, 0.2],
                    'num_leaves': [31, 50, 100]
                }
            }
        }
    
    tuned_results = []
    
    for name, model_info in models_params.items():
        print(f"\n🔄 Tuning {name}...")
        
        # Grid search with cross-validation
        grid_search = GridSearchCV(
            model_info['model'], 
            model_info['params'],
            cv=5, 
            scoring=scoring,
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train, y_train)
            
            # Best model evaluation
            best_model = grid_search.best_estimator_
            train_score = best_model.score(X_train, y_train)
            test_score = best_model.score(X_test, y_test)
            
            tuned_results.append({
                'Model': name,
                'Best_Params': str(grid_search.best_params_),
                'Best_CV_Score': grid_search.best_score_,
                'Train_Score': train_score,
                'Test_Score': test_score,
                'Best_Model': best_model
            })
            
            print(f"✅ {name}: Best {scoring} = {test_score:.4f}")
            print(f"   Best params: {grid_search.best_params_}")
            
        except Exception as e:
            print(f"❌ {name}: Error - {str(e)}")
    
    return pd.DataFrame(tuned_results).sort_values('Test_Score', ascending=False)

# ============================================================================
# SECTION 6: FEATURE IMPORTANCE & EXPLAINABILITY
# ============================================================================

def analyze_feature_importance(X, y, model=None):
    """Analyze feature importance using various methods"""
    print("\n🔍 FEATURE IMPORTANCE ANALYSIS")
    print("-" * 40)
    
    feature_names = X.columns.tolist()
    
    if model is None:
        # Use Random Forest as default
        model = RandomForestRegressor(n_estimators=100, random_state=42)
        model.fit(X, y)
    
    # 1. Model-based feature importance
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        importance_df = pd.DataFrame({
            'Feature': feature_names,
            'Importance': importances
        }).sort_values('Importance', ascending=False)
        
        print("📊 Top 10 Most Important Features:")
        print(importance_df.head(10))
    
    # 2. Correlation with target
    correlations = X.corrwith(y).abs().sort_values(ascending=False)
    correlation_df = pd.DataFrame({
        'Feature': correlations.index,
        'Correlation': correlations.values
    })
    
    print("\n📈 Top 10 Features by Correlation with Target:")
    print(correlation_df.head(10))
    
    return importance_df if 'importance_df' in locals() else None, correlation_df

# ============================================================================
# SECTION 7: MODEL DEPLOYMENT PIPELINE
# ============================================================================

def create_deployment_pipeline(best_model, scaler, feature_columns):
    """Create a deployment-ready pipeline"""
    print("\n🚀 DEPLOYMENT PIPELINE")
    print("-" * 30)
    
    class FinancialInclusionPredictor:
        def __init__(self, model, scaler, features):
            self.model = model
            self.scaler = scaler
            self.features = features
        
        def predict(self, X_new):
            """Make predictions on new data"""
            # Ensure correct feature order
            X_new = X_new[self.features]
            
            # Scale if needed
            if self.scaler is not None:
                X_new = self.scaler.transform(X_new)
            
            # Predict
            predictions = self.model.predict(X_new)
            
            # Get probabilities if classifier
            if hasattr(self.model, 'predict_proba'):
                probabilities = self.model.predict_proba(X_new)
                return predictions, probabilities
            else:
                return predictions
        
        def get_feature_importance(self):
            """Get feature importance"""
            if hasattr(self.model, 'feature_importances_'):
                return dict(zip(self.features, self.model.feature_importances_))
            else:
                return None
    
    predictor = FinancialInclusionPredictor(best_model, scaler, feature_columns)
    
    print("✅ Deployment pipeline created successfully!")
    print("📋 Usage:")
    print("   predictor.predict(new_data)")
    print("   predictor.get_feature_importance()")
    
    return predictor

# ============================================================================
# SECTION 8: MAIN EXECUTION FUNCTION
# ============================================================================

def run_complete_ml_pipeline(df):
    """Run the complete ML pipeline"""
    print("🎯 STARTING COMPLETE ML PIPELINE")
    print("="*70)
    
    # 1. Data Preparation
    X, y, cat_cols, num_cols, encoders = prepare_data(df)
    
    # 2. Regression Models
    regression_results, reg_scaler = evaluate_regression_models(X, y)
    print("\n🏆 REGRESSION RESULTS:")
    print(regression_results[['Model', 'Test_R2', 'Test_RMSE', 'CV_R2_Mean']].to_string(index=False))
    
    # 3. Classification Models
    classification_results = evaluate_classification_models(X, y, threshold=0.5)
    print("\n🏆 CLASSIFICATION RESULTS:")
    print(classification_results[['Model', 'Test_Accuracy', 'F1_Score', 'ROC_AUC']].to_string(index=False))
    
    # 4. Clustering Analysis
    clustering_results = perform_clustering_analysis(X, y)
    
    # 5. Hyperparameter Tuning
    tuned_reg_results = hyperparameter_tuning(X, y, 'regression')
    tuned_clf_results = hyperparameter_tuning(X, y, 'classification')
    
    print("\n🏆 TUNED REGRESSION RESULTS:")
    print(tuned_reg_results[['Model', 'Test_Score', 'Best_CV_Score']].to_string(index=False))
    
    print("\n🏆 TUNED CLASSIFICATION RESULTS:")
    print(tuned_clf_results[['Model', 'Test_Score', 'Best_CV_Score']].to_string(index=False))
    
    # 6. Feature Importance
    best_reg_model = tuned_reg_results.iloc[0]['Best_Model']
    importance_df, correlation_df = analyze_feature_importance(X, y, best_reg_model)
    
    # 7. Create Deployment Pipeline
    predictor = create_deployment_pipeline(best_reg_model, reg_scaler, X.columns.tolist())
    
    # 8. Final Recommendations
    print("\n" + "="*70)
    print("📋 FINAL RECOMMENDATIONS & IMPLEMENTATION GUIDE")
    print("="*70)
    
    print("\n🥇 BEST MODELS:")
    print(f"   • Regression: {tuned_reg_results.iloc[0]['Model']} (R² = {tuned_reg_results.iloc[0]['Test_Score']:.4f})")
    print(f"   • Classification: {tuned_clf_results.iloc[0]['Model']} (Accuracy = {tuned_clf_results.iloc[0]['Test_Score']:.4f})")
    
    print(f"\n🔧 IMPLEMENTATION STEPS:")
    print("   1. Use the tuned models for production")
    print("   2. Monitor model performance over time")
    print("   3. Retrain periodically with new data")
    print("   4. Use clustering results for customer segmentation")
    print("   5. Focus on top features for policy interventions")
    
    if importance_df is not None:
        print(f"\n🎯 TOP 5 FEATURES TO FOCUS ON:")
        for i, row in importance_df.head(5).iterrows():
            print(f"   • {row['Feature']}: {row['Importance']:.4f}")
    
    return {
        'regression_results': regression_results,
        'classification_results': classification_results,
        'tuned_regression': tuned_reg_results,
        'tuned_classification': tuned_clf_results,
        'clustering_results': clustering_results,
        'feature_importance': importance_df,
        'predictor': predictor,
        'encoders': encoders
    }


🚀 COMPREHENSIVE ML PIPELINE FOR FINANCIAL INCLUSION PREDICTION


In [42]:
results = run_complete_ml_pipeline(df_modeling)

🎯 STARTING COMPLETE ML PIPELINE

📊 DATA PREPARATION
------------------------------
✅ Features: 29
✅ Categorical features: 2 - ['demo_subgroup', 'region_cleaned']
✅ Numerical features: 27
✅ Target range: 0.004 to 1.000

🎯 REGRESSION MODELS EVALUATION
----------------------------------------

🔄 Training Linear Regression...
✅ Linear Regression: Test R² = 0.6581, RMSE = 0.1657

🔄 Training Ridge Regression...
✅ Ridge Regression: Test R² = 0.6583, RMSE = 0.1657

🔄 Training Lasso Regression...
✅ Lasso Regression: Test R² = 0.5308, RMSE = 0.1941

🔄 Training ElasticNet...
✅ ElasticNet: Test R² = 0.5974, RMSE = 0.1798

🔄 Training Random Forest...
✅ Random Forest: Test R² = 0.8503, RMSE = 0.1097

🔄 Training Gradient Boosting...
✅ Gradient Boosting: Test R² = 0.8224, RMSE = 0.1194

🔄 Training XGBoost...
✅ XGBoost: Test R² = 0.8614, RMSE = 0.1055

🔄 Training LightGBM...
✅ LightGBM: Test R² = 0.8588, RMSE = 0.1065

🔄 Training SVR...
✅ SVR: Test R² = 0.8470, RMSE = 0.1109

🔄 Training KNN...
✅ KNN: T

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error, 
                           accuracy_score, precision_score, recall_score, f1_score, 
                           roc_auc_score, classification_report, confusion_matrix)
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# SECTION 1: DATA PREPARATION AND EXPLORATION
# ============================================================================
def prepare_data(df):
    """Prepare data for ML modeling"""
    print("\n" + "="*80)
    print("📊 SECTION 1: DATA PREPARATION AND EXPLORATION")
    print("="*80)
    
    # Separate features and target
    target_col = 'has_account'
    feature_cols = [col for col in df.columns if col != target_col]
    
    X = df[feature_cols].copy()
    y = df[target_col].copy()
    
    # Identify feature types
    numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = []  # Already encoded in feature engineering
    
    print(f"✅ Total Features: {len(feature_cols)}")
    print(f"✅ Numerical features: {len(numerical_cols)}")
    print(f"✅ Target range: {y.min():.3f} to {y.max():.3f}")
    print(f"✅ Target mean: {y.mean():.3f}")
    print(f"✅ Data shape: {df.shape}")
    
    # Feature categories
    protected_features = [col for col in X.columns if 
                         ('encoded' in col or 'score' in col or 'interaction' in col or
                          'high_' in col or 'digital_native' in col)]
    engineered_features = [col for col in X.columns if 
                          ('score' in col or 'interaction' in col or 'high_' in col or 'digital_native' in col)]
    
    print(f"✅ Protected features: {len(protected_features)}")
    print(f"✅ Engineered features: {len(engineered_features)}")
    
    return X, y, categorical_cols, numerical_cols, protected_features, engineered_features

# ============================================================================
# SECTION 2: REGRESSION MODELS (Predicting Continuous Account Probability)
# ============================================================================
def evaluate_baseline_regression_models(X, y):
    """Evaluate baseline regression models with default parameters"""
    print("\n" + "="*80)
    print("🎯 SECTION 2A: BASELINE REGRESSION MODELS")
    print("="*80)
    print("Using default parameters and basic preprocessing")
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Basic scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Baseline models with default parameters
    baseline_models = {
        'Linear Regression': LinearRegression(),
        'Ridge (default)': Ridge(),
        'Lasso (default)': Lasso(),
        'Random Forest (default)': RandomForestRegressor(random_state=42),
        'Decision Tree': DecisionTreeRegressor(random_state=42),
        'KNN (default)': KNeighborsRegressor()
    }
    
    baseline_results = []
    
    print(f"\n🔄 Training {len(baseline_models)} baseline models...")
    
    for name, model in baseline_models.items():
        try:
            # Use scaled data for models that need it
            if 'KNN' in name or 'Linear' in name or 'Ridge' in name or 'Lasso' in name:
                X_train_model = X_train_scaled
                X_test_model = X_test_scaled
            else:
                X_train_model = X_train
                X_test_model = X_test
            
            # Train model
            model.fit(X_train_model, y_train)
            
            # Predictions
            y_pred_train = model.predict(X_train_model)
            y_pred_test = model.predict(X_test_model)
            
            # Metrics
            train_r2 = r2_score(y_train, y_pred_train)
            test_r2 = r2_score(y_test, y_pred_test)
            train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
            test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
            test_mae = mean_absolute_error(y_test, y_pred_test)
            
            # Cross-validation
            cv_scores = cross_val_score(model, X_train_model, y_train, cv=5, scoring='r2')
            
            baseline_results.append({
                'Model': name,
                'Type': 'Baseline',
                'Train_R2': train_r2,
                'Test_R2': test_r2,
                'Train_RMSE': train_rmse,
                'Test_RMSE': test_rmse,
                'Test_MAE': test_mae,
                'CV_R2_Mean': cv_scores.mean(),
                'CV_R2_Std': cv_scores.std(),
                'Overfitting': train_r2 - test_r2
            })
            
            print(f"✅ {name:<25}: Test R² = {test_r2:.4f}, RMSE = {test_rmse:.4f}")
            
        except Exception as e:
            print(f"❌ {name}: Error - {str(e)}")
    
    baseline_df = pd.DataFrame(baseline_results).sort_values('Test_R2', ascending=False)
    
    print(f"\n📊 BASELINE REGRESSION RESULTS:")
    print(f"Best Model: {baseline_df.iloc[0]['Model']} (R² = {baseline_df.iloc[0]['Test_R2']:.4f})")
    
    return baseline_df, scaler


def evaluate_improved_regression_models(X, y):
    """Evaluate improved regression models with hyperparameter tuning"""
    print("\n" + "="*80)
    print("🚀 SECTION 2B: IMPROVED REGRESSION MODELS")
    print("="*80)
    print("Using hyperparameter tuning and advanced algorithms")
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Advanced scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Improved models with better parameters
    improved_models = {
        'Ridge (tuned)': Ridge(alpha=10.0),
        'Lasso (tuned)': Lasso(alpha=0.001, max_iter=2000),
        'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=2000),
        'Random Forest (tuned)': RandomForestRegressor(
            n_estimators=200, max_depth=10, min_samples_split=5, 
            min_samples_leaf=2, random_state=42
        ),
        'Gradient Boosting': GradientBoostingRegressor(
            n_estimators=200, learning_rate=0.1, max_depth=6, random_state=42
        ),
        'XGBoost': xgb.XGBRegressor(
            n_estimators=200, learning_rate=0.1, max_depth=6, 
            random_state=42, eval_metric='rmse'
        ),
        'LightGBM': lgb.LGBMRegressor(
            n_estimators=200, learning_rate=0.1, max_depth=6, 
            random_state=42, verbose=-1
        ),
        'SVR (tuned)': SVR(kernel='rbf', C=10.0, gamma='scale'),
        'Neural Network': MLPRegressor(
            hidden_layer_sizes=(200, 100, 50), alpha=0.01, 
            learning_rate_init=0.001, max_iter=1000, random_state=42
        )
    }
    
    improved_results = []
    
    print(f"\n🔄 Training {len(improved_models)} improved models...")
    
    for name, model in improved_models.items():
        try:
            # Use scaled data for models that need it
            if any(keyword in name for keyword in ['Ridge', 'Lasso', 'ElasticNet', 'SVR', 'Neural']):
                X_train_model = X_train_scaled
                X_test_model = X_test_scaled
            else:
                X_train_model = X_train
                X_test_model = X_test
            
            # Train model
            model.fit(X_train_model, y_train)
            
            # Predictions
            y_pred_train = model.predict(X_train_model)
            y_pred_test = model.predict(X_test_model)
            
            # Metrics
            train_r2 = r2_score(y_train, y_pred_train)
            test_r2 = r2_score(y_test, y_pred_test)
            train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
            test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
            test_mae = mean_absolute_error(y_test, y_pred_test)
            
            # Cross-validation
            cv_scores = cross_val_score(model, X_train_model, y_train, cv=5, scoring='r2')
            
            improved_results.append({
                'Model': name,
                'Type': 'Improved',
                'Train_R2': train_r2,
                'Test_R2': test_r2,
                'Train_RMSE': train_rmse,
                'Test_RMSE': test_rmse,
                'Test_MAE': test_mae,
                'CV_R2_Mean': cv_scores.mean(),
                'CV_R2_Std': cv_scores.std(),
                'Overfitting': train_r2 - test_r2
            })
            
            print(f"✅ {name:<25}: Test R² = {test_r2:.4f}, RMSE = {test_rmse:.4f}")
            
        except Exception as e:
            print(f"❌ {name}: Error - {str(e)}")
    
    improved_df = pd.DataFrame(improved_results).sort_values('Test_R2', ascending=False)
    
    print(f"\n📊 IMPROVED REGRESSION RESULTS:")
    print(f"Best Model: {improved_df.iloc[0]['Model']} (R² = {improved_df.iloc[0]['Test_R2']:.4f})")
    
    return improved_df


def compare_regression_results(baseline_df, improved_df):
    """Compare baseline vs improved regression results"""
    print("\n" + "="*80)
    print("📈 SECTION 2C: REGRESSION COMPARISON ANALYSIS")
    print("="*80)
    
    # Combine results
    combined_df = pd.concat([baseline_df, improved_df], ignore_index=True)
    
    # Display top performers
    top_results = combined_df.nlargest(10, 'Test_R2')[['Model', 'Type', 'Test_R2', 'Test_RMSE', 'CV_R2_Mean', 'Overfitting']]
    
    print("🏆 TOP 10 REGRESSION MODELS:")
    print(top_results.to_string(index=False, float_format='{:.4f}'.format))
    
    # Best baseline vs best improved
    best_baseline = baseline_df.iloc[0]
    best_improved = improved_df.iloc[0]
    
    improvement = best_improved['Test_R2'] - best_baseline['Test_R2']
    
    print(f"\n🎯 PERFORMANCE COMPARISON:")
    print(f"Best Baseline:  {best_baseline['Model']:<25} R² = {best_baseline['Test_R2']:.4f}")
    print(f"Best Improved:  {best_improved['Model']:<25} R² = {best_improved['Test_R2']:.4f}")
    print(f"Improvement:    {improvement:+.4f} R² points ({improvement/best_baseline['Test_R2']*100:+.1f}%)")
    
    return combined_df


# ============================================================================
# SECTION 3: CLASSIFICATION MODELS (Convert to Binary Classification)
# ============================================================================
def evaluate_baseline_classification_models(X, y, threshold=0.5):
    """Evaluate baseline classification models"""
    print("\n" + "="*80)
    print("🎯 SECTION 3A: BASELINE CLASSIFICATION MODELS")
    print("="*80)
    print(f"Using threshold = {threshold} for binary classification")
    
    # Convert continuous target to binary
    y_binary = (y >= threshold).astype(int)
    class_dist = y_binary.value_counts()
    print(f"✅ Class distribution: No Account = {class_dist[0]}, Has Account = {class_dist[1]}")
    print(f"✅ Positive class ratio: {class_dist[1]/len(y_binary):.1%}")
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y_binary, test_size=0.2, 
                                                        random_state=42, stratify=y_binary)
    
    # Basic scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Baseline classification models
    baseline_models = {
        'Logistic Regression': LogisticRegression(random_state=42),
        'Random Forest (default)': RandomForestClassifier(random_state=42),
        'Decision Tree': DecisionTreeClassifier(random_state=42),
        'KNN (default)': KNeighborsClassifier()
    }
    
    baseline_results = []
    
    print(f"\n🔄 Training {len(baseline_models)} baseline classification models...")
    
    for name, model in baseline_models.items():
        try:
            # Use scaled data for models that need it
            if 'Logistic' in name or 'KNN' in name:
                X_train_model = X_train_scaled
                X_test_model = X_test_scaled
            else:
                X_train_model = X_train
                X_test_model = X_test
            
            # Train model
            model.fit(X_train_model, y_train)
            
            # Predictions
            y_pred_train = model.predict(X_train_model)
            y_pred_test = model.predict(X_test_model)
            y_pred_proba = model.predict_proba(X_test_model)[:, 1] if hasattr(model, 'predict_proba') else None
            
            # Metrics
            train_acc = accuracy_score(y_train, y_pred_train)
            test_acc = accuracy_score(y_test, y_pred_test)
            precision = precision_score(y_test, y_pred_test)
            recall = recall_score(y_test, y_pred_test)
            f1 = f1_score(y_test, y_pred_test)
            auc = roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else None
            
            # Cross-validation
            cv_scores = cross_val_score(model, X_train_model, y_train, cv=5, scoring='accuracy')
            
            baseline_results.append({
                'Model': name,
                'Type': 'Baseline',
                'Train_Accuracy': train_acc,
                'Test_Accuracy': test_acc,
                'Precision': precision,
                'Recall': recall,
                'F1_Score': f1,
                'ROC_AUC': auc,
                'CV_Accuracy_Mean': cv_scores.mean(),
                'CV_Accuracy_Std': cv_scores.std(),
                'Overfitting': train_acc - test_acc
            })
            
            print(f"✅ {name:<25}: Acc = {test_acc:.4f}, F1 = {f1:.4f}, AUC = {auc:.4f if auc else 'N/A'}")
            
        except Exception as e:
            print(f"❌ {name}: Error - {str(e)}")
    
    baseline_df = pd.DataFrame(baseline_results).sort_values('Test_Accuracy', ascending=False)
    
    print(f"\n📊 BASELINE CLASSIFICATION RESULTS:")
    if not baseline_df.empty:
        print(f"Best Model: {baseline_df.iloc[0]['Model']} (Accuracy = {baseline_df.iloc[0]['Test_Accuracy']:.4f})")
    
    return baseline_df


def evaluate_improved_classification_models(X, y, threshold=0.5):
    """Evaluate improved classification models with hyperparameter tuning"""
    print("\n" + "="*80)
    print("🚀 SECTION 3B: IMPROVED CLASSIFICATION MODELS")
    print("="*80)
    print("Using hyperparameter tuning and advanced algorithms")
    
    # Convert continuous target to binary
    y_binary = (y >= threshold).astype(int)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y_binary, test_size=0.2, 
                                                        random_state=42, stratify=y_binary)
    
    # Advanced scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Improved classification models
    improved_models = {
        'Logistic Regression (tuned)': LogisticRegression(
            C=10.0, penalty='l2', solver='liblinear', random_state=42, max_iter=2000
        ),
        'Random Forest (tuned)': RandomForestClassifier(
            n_estimators=200, max_depth=10, min_samples_split=5, 
            min_samples_leaf=2, class_weight='balanced', random_state=42
        ),
        'Gradient Boosting': GradientBoostingClassifier(
            n_estimators=200, learning_rate=0.1, max_depth=6, random_state=42
        ),
        'XGBoost': xgb.XGBClassifier(
            n_estimators=200, learning_rate=0.1, max_depth=6, 
            scale_pos_weight=1, random_state=42, eval_metric='logloss'
        ),
        'LightGBM': lgb.LGBMClassifier(
            n_estimators=200, learning_rate=0.1, max_depth=6, 
            class_weight='balanced', random_state=42, verbose=-1
        ),
        'SVM (tuned)': SVC(
            kernel='rbf', C=10.0, gamma='scale', probability=True, 
            class_weight='balanced', random_state=42
        ),
        'Neural Network': MLPClassifier(
            hidden_layer_sizes=(200, 100, 50), alpha=0.01, 
            learning_rate_init=0.001, max_iter=1000, random_state=42
        )
    }
    
    improved_results = []
    
    print(f"\n🔄 Training {len(improved_models)} improved classification models...")
    
    for name, model in improved_models.items():
        try:
            # Use scaled data for models that need it
            if any(keyword in name for keyword in ['Logistic', 'SVM', 'Neural']):
                X_train_model = X_train_scaled
                X_test_model = X_test_scaled
            else:
                X_train_model = X_train
                X_test_model = X_test
            
            # Train model
            model.fit(X_train_model, y_train)
            
            # Predictions
            y_pred_train = model.predict(X_train_model)
            y_pred_test = model.predict(X_test_model)
            y_pred_proba = model.predict_proba(X_test_model)[:, 1] if hasattr(model, 'predict_proba') else None
            
            # Metrics
            train_acc = accuracy_score(y_train, y_pred_train)
            test_acc = accuracy_score(y_test, y_pred_test)
            precision = precision_score(y_test, y_pred_test, zero_division=0)
            recall = recall_score(y_test, y_pred_test, zero_division=0)
            f1 = f1_score(y_test, y_pred_test, zero_division=0)
            auc = roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else None
            
            # Cross-validation
            cv_scores = cross_val_score(model, X_train_model, y_train, cv=5, scoring='accuracy')
            
            improved_results.append({
                'Model': name,
                'Type': 'Improved',
                'Train_Accuracy': train_acc,
                'Test_Accuracy': test_acc,
                'Precision': precision,
                'Recall': recall,
                'F1_Score': f1,
                'ROC_AUC': auc,
                'CV_Accuracy_Mean': cv_scores.mean(),
                'CV_Accuracy_Std': cv_scores.std(),
                'Overfitting': train_acc - test_acc
            })
            
            print(f"✅ {name:<25}: Acc = {test_acc:.4f}, F1 = {f1:.4f}, AUC = {auc:.4f if auc else 'N/A'}")
            
        except Exception as e:
            print(f"❌ {name}: Error - {str(e)}")
    
    improved_df = pd.DataFrame(improved_results).sort_values('Test_Accuracy', ascending=False)
    
    print(f"\n📊 IMPROVED CLASSIFICATION RESULTS:")
    if not improved_df.empty:
        print(f"Best Model: {improved_df.iloc[0]['Model']} (Accuracy = {improved_df.iloc[0]['Test_Accuracy']:.4f})")
    
    return improved_df


def compare_classification_results(baseline_df, improved_df):
    """Compare baseline vs improved classification results"""
    print("\n" + "="*80)
    print("📈 SECTION 3C: CLASSIFICATION COMPARISON ANALYSIS")
    print("="*80)
    
    # Combine results
    combined_df = pd.concat([baseline_df, improved_df], ignore_index=True)
    
    # Display top performers
    top_results = combined_df.nlargest(10, 'Test_Accuracy')[['Model', 'Type', 'Test_Accuracy', 'F1_Score', 'ROC_AUC', 'Overfitting']]
    
    print("🏆 TOP 10 CLASSIFICATION MODELS:")
    print(top_results.to_string(index=False, float_format='{:.4f}'.format))
    
    # Best baseline vs best improved
    if not baseline_df.empty and not improved_df.empty:
        best_baseline = baseline_df.iloc[0]
        best_improved = improved_df.iloc[0]
        
        acc_improvement = best_improved['Test_Accuracy'] - best_baseline['Test_Accuracy']
        f1_improvement = best_improved['F1_Score'] - best_baseline['F1_Score']
        
        print(f"\n🎯 PERFORMANCE COMPARISON:")
        print(f"Best Baseline:  {best_baseline['Model']:<30} Acc = {best_baseline['Test_Accuracy']:.4f}")
        print(f"Best Improved:  {best_improved['Model']:<30} Acc = {best_improved['Test_Accuracy']:.4f}")
        print(f"Accuracy Improvement:  {acc_improvement:+.4f} points ({acc_improvement/best_baseline['Test_Accuracy']*100:+.1f}%)")
        print(f"F1 Score Improvement:  {f1_improvement:+.4f} points ({f1_improvement/best_baseline['F1_Score']*100:+.1f}%)")
    
    return combined_df


# ============================================================================
# SECTION 4: MAIN EXECUTION FUNCTION
# ============================================================================
def run_complete_ml_pipeline(df):
    """Run the complete ML pipeline with baseline vs improved comparison"""
    print("\n" + "🤖"*40)
    print("🚀 COMPLETE ML PIPELINE: BASELINE vs IMPROVED MODELS")
    print("🤖"*40)
    
    # Prepare data
    X, y, categorical_cols, numerical_cols, protected_features, engineered_features = prepare_data(df)
    
    # REGRESSION ANALYSIS
    baseline_reg_results, scaler = evaluate_baseline_regression_models(X, y)
    improved_reg_results = evaluate_improved_regression_models(X, y)
    combined_reg_results = compare_regression_results(baseline_reg_results, improved_reg_results)
    
    # CLASSIFICATION ANALYSIS
    baseline_clf_results = evaluate_baseline_classification_models(X, y, threshold=0.5)
    improved_clf_results = evaluate_improved_classification_models(X, y, threshold=0.5)
    combined_clf_results = compare_classification_results(baseline_clf_results, improved_clf_results)
    
    # FINAL SUMMARY
    print("\n" + "="*80)
    print("🏁 FINAL PIPELINE SUMMARY")
    print("="*80)
    
    print(f"📊 Dataset: {df.shape[0]} samples, {df.shape[1]} features")
    print(f"🔧 Feature Engineering: {len(engineered_features)} engineered features")
    
    if not combined_reg_results.empty:
        best_reg = combined_reg_results.iloc[0]
        print(f"🏆 Best Regression Model: {best_reg['Model']} (R² = {best_reg['Test_R2']:.4f})")
    
    if not combined_clf_results.empty:
        best_clf = combined_clf_results.iloc[0]
        print(f"🏆 Best Classification Model: {best_clf['Model']} (Acc = {best_clf['Test_Accuracy']:.4f})")
    
    return {
        'regression_results': combined_reg_results,
        'classification_results': combined_clf_results,
        'feature_info': {
            'protected_features': protected_features,
            'engineered_features': engineered_features,
            'total_features': len(X.columns)
        }
    }

In [44]:
regression_models = results['regression_results']
regression_models

,Model,Train_R2,Test_R2,Train_RMSE,Test_RMSE,Test_MAE,CV_R2_Mean,CV_R2_Std,Overfitting
6,XGBoost,0.939561,0.861408,0.068995,0.105509,0.072309,0.863343,0.008164,0.078153
7,LightGBM,0.904967,0.858772,0.086516,0.106508,0.073888,0.866876,0.007257,0.046195
4,Random Forest,0.946489,0.850300,0.064920,0.109656,0.073441,0.854583,0.009616,0.096189
10,Neural Network,0.887559,0.848210,0.094107,0.110418,0.077635,0.847742,0.009944,0.039349
8,SVR,0.867534,0.846993,0.102143,0.110860,0.080955,0.846846,0.003712,0.020542
5,Gradient Boosting,0.848976,0.822380,0.109064,0.119444,0.086776,0.835193,0.005236,0.026596
9,KNN,0.870387,0.814412,0.101037,0.122094,0.086303,0.807352,0.011827,0.055974
11,Decision Tree,0.959085,0.758011,0.056768,0.139417,0.092308,0.759132,0.007630,0.201073
1,Ridge Regression,0.669400,0.658304,0.161365,0.165668,0.130716,0.666318,0.010696,0.011096
0,Linear Regression,0.669539,0.658149,0.161331,0.165706,0.130677,0.666489,0.010752,0.011390


In [45]:
classification_models = results['classification_results']
classification_models

,Model,Train_Accuracy,Test_Accuracy,Precision,Recall,F1_Score,ROC_AUC,CV_Accuracy_Mean,CV_Accuracy_Std,Overfitting
1,Random Forest,0.988201,0.913325,0.925996,0.933971,0.929967,0.967849,0.912979,0.008304,0.074875
4,LightGBM,0.969174,0.913325,0.934236,0.924402,0.929293,0.975755,0.912242,0.006546,0.055849
3,XGBoost,0.982153,0.908019,0.923737,0.927273,0.925501,0.974771,0.910619,0.006987,0.074135
7,Neural Network,0.958702,0.902123,0.935580,0.903349,0.919182,0.972743,0.900147,0.006020,0.056579
2,Gradient Boosting,0.911504,0.892689,0.914505,0.911005,0.912752,0.962077,0.895280,0.011145,0.018816
5,SVM,0.908112,0.883844,0.905354,0.906220,0.905787,0.956093,0.893805,0.010948,0.024268
6,KNN,0.915929,0.876179,0.900288,0.898565,0.899425,0.939288,0.869912,0.003478,0.039750
8,Decision Tree,0.988201,0.873231,0.892250,0.903349,0.897765,0.875519,0.866814,0.005911,0.114969
0,Logistic Regression,0.869469,0.862618,0.885199,0.892823,0.888995,0.929801,0.865929,0.011922,0.006851


In [46]:
best_tuned_models = results['tuned_regression']
best_tuned_models

,Model,Best_Params,Best_CV_Score,Train_Score,Test_Score,Best_Model
2,LightGBM,"{'learning_rate': 0.1, 'max_depth': 10, 'n_est...",0.871126,0.931750,0.866948,"LGBMRegressor(max_depth=10, n_estimators=200, ..."
1,XGBoost,"{'learning_rate': 0.1, 'max_depth': 6, 'n_esti...",0.868163,0.930371,0.861181,"XGBRegressor(base_score=None, booster=None, ca..."
0,Random Forest,"{'max_depth': None, 'min_samples_leaf': 2, 'mi...",0.855405,0.937741,0.850251,"(DecisionTreeRegressor(max_features=1.0, min_s..."


In [47]:
clustering_segments = results['clustering_results']
clustering_segments

{'kmeans_model': KMeans(n_clusters=2, random_state=42),
 'kmeans_labels': array([1, 1, 1, ..., 0, 0, 0], dtype=int32),
 'dbscan_model': DBSCAN(eps=0.3),
 'dbscan_labels': array([-1,  0, -1, ..., -1, -1, -1]),
 'scaler': StandardScaler(),
 'cluster_analysis':       demo_subgroup  borrowed_any  biz_loan_source  biz_loan  \
 0                 2      0.000000         0.028194  0.028194   
 1                 2      0.000000         0.085616  0.085616   
 2                 2      0.000000         0.043257  0.043257   
 3                 2      0.000000         0.159156  0.159156   
 4                 2      0.000000         0.038007  0.038007   
 ...             ...           ...              ...       ...   
 8471              7      0.659608         0.195387  0.101581   
 8472             10      0.609592         0.375945  0.302609   
 8473              7      0.636481         0.163775  0.130235   
 8474             10      0.582794         0.600300  0.581244   
 8475              7      0

In [48]:
feature_rankings = results['feature_importance']
feature_rankings

,Feature,Importance
2,biz_loan_source,1133
6,saved_any,1083
1,borrowed_any,951
26,income_digital_interaction,938
4,saved_for_purchase,934
25,financial_activity_score,815
3,biz_loan,790
0,demo_subgroup,525
5,saved_no_purpose,487
7,region_cleaned,479


In [49]:
deployment_model = results['predictor']
deployment_model

<__main__.create_deployment_pipeline.<locals>.FinancialInclusionPredictor at 0x13d0f7770>

In [53]:
predictions = results['predictor'].predict(df_cleaned.iloc[:5])
predictions


KeyError: "['region_encoded', 'income_group_encoded', 'demo_group_encoded', 'financial_activity_score', 'income_digital_interaction', 'high_financial_activity', 'digital_native'] not in index"

Exception ignored in: <function ResourceTracker.__del__ at 0x10728db20>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x107601b20>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x10cd45b20>
Traceback (most recent call last

In [40]:
df_MODELING.columns

Index(['demo_subgroup', 'has_account', 'borrowed_any', 'biz_loan_source',
       'biz_loan', 'saved_for_purchase', 'saved_no_purpose', 'saved_any',
       'region_cleaned', 'biz_loan_source_missing', 'biz_loan_missing',
       'loan_purpose_group_missing', 'loan_purpose_missing',
       'saved_for_purchase_missing', 'saved_no_purpose_missing',
       'digital_payment_other_missing', 'mobile_payment_bill_missing',
       'govt_digital_pay_acc_missing', 'digital_pay_missing',
       'digital_pay_acc_missing', 'govt_payment_recv_missing',
       'prefer_digital_fin_missing', 'prefer_digital_missing',
       'region_encoded', 'income_group_encoded', 'demo_group_encoded',
       'financial_activity_score', 'income_digital_interaction',
       'high_financial_activity', 'digital_native'],
      dtype='object')